# HDB Rentals 2: models

Report section: Data Mining Methods. Run after `1_eda_preprocessing`. Parts 2.1 to 2.3 are the tabular models; Part 2.4 trains the GNNs.

Each part was its own notebook; text inside a part may still say "notebook NN". The originals are in `notebooks/archive/`.

| Part | Was |
|---|---|
| 2.1 | `07_pipeline.ipynb` |
| 2.2 | `09_model_training.ipynb` |
| 2.3 | `10_external_rentals.ipynb` |
| 2.4 | `scripts/*gnn*.py` |

## Part 2.1. HDB Rentals: progress-report pipeline

This notebook reproduces the progress report's findings as of the checkpoint (slides 4 to 13), in one place
that runs on this branch alone, and then does next steps 3 and 4: combine location features with the
market-level model, and write a first submission. Nothing here goes beyond the report; work done after the
checkpoint (nine-origin level check, model comparison, tuning) is in `08_level_robustness.ipynb`.

| section | slide | source |
|---|---|---|
| 1. Clean the rental data | 4 | `main.ipynb` §1 (summary in cell 44), `01_eda_train` §4 |
| 2. Block-level location features | 5, 12 | `01_eda_hdb`/`mrt`/`school`, `06_distance_features` §2, §4e (`auxiliary-distance-features`) |
| 3. Validation: five walk-forward folds + one 16-month fold | 8 | `macro-time-trend`: `src/hdb_rentals/validation.py` |
| 4. Baseline and the horizon problem | 9 | `macro-time-trend`: `baselines.py`, `experiments.jsonl` |
| 5. Location features | 12 | `06_distance_features` §4e |
| 6. Market level × flat premium | 11 | `macro-time-trend`: `market_level.py` |
| 7. Macro data | 10 | `macro-time-trend`: `macro.py` (not re-run) |
| 8. Combined model | next step 3 | this notebook |
| 9. Fit on all of train, first submission | next step 4 | this notebook |
| 10. Summary | 13 | |

The `macro-time-trend` helpers are rewritten inline (same logic, no `src` import), so the notebook needs only
`data/`, `cleaned/` and the auxiliary files.

In [ ]:
import json
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from statsmodels.tsa.holtwinters import ExponentialSmoothing

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA, CLEANED, OUT = ROOT / "data", ROOT / "cleaned", ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
TARGET = "MONTHLY_RENT"
KEY = ["TOWN", "BLOCK", "STREET"]
SEED = 0
T0 = time.time()

### 1. Clean the rental data (slide 4)

Train and test go through the same function:

- lowercase and strip every text column; "4 room" and "4-room" become one level, so 11 flat-type labels collapse
  to 6, ordinal-encoded 1 to 6 as `FLAT_TYPE_ORD`;
- drop the constant `FURNISHED` and `FEE`;
- add `FLAT_AGE` (approval year minus lease start) and the month counter `MONTH_INDEX` (2021-01 = 0);
- `STREET` is kept and `BLOCK` dropped from the model inputs (576 streets vs 9,406 block + street pairs). `BLOCK`
  is still used as part of the join key in section 2.

Train is sorted by month (stable), as `data.load_train` does on `macro-time-trend`, so rows reach LightGBM in the
same order as in the logged runs. Test keeps the file's order, since the submission is matched by row position.

In [ ]:
FLAT_TYPE_ORDER = ["1-room", "2-room", "3-room", "4-room", "5-room", "executive"]


def lower_strip(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    text = [c for c in out.columns if pd.api.types.is_string_dtype(out[c])]
    out[text] = out[text].apply(lambda s: s.str.strip().str.lower())
    return out


def clean_rent(df: pd.DataFrame) -> pd.DataFrame:
    out = lower_strip(df).drop(columns=["FURNISHED", "FEE"])
    # "4 room" and "4-room" are the same level.
    out["FLAT_TYPE"] = out["FLAT_TYPE"].str.replace(r"[\s\-]+", "-", regex=True)
    date = pd.to_datetime(out["RENT_APPROVAL_DATE"], format="%Y-%m")
    out["RENT_APPROVAL_DATE"] = date
    out["MONTH_INDEX"] = (date.dt.year - 2021) * 12 + date.dt.month - 1
    out["FLAT_AGE"] = date.dt.year - out["LEASE_COMMENCE_DATE"]
    out["FLAT_TYPE_ORD"] = out["FLAT_TYPE"].map({t: i + 1 for i, t in enumerate(FLAT_TYPE_ORDER)})
    return out


raw_train, raw_test = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
train = clean_rent(raw_train).sort_values("MONTH_INDEX", kind="stable").reset_index(drop=True)
test = clean_rent(raw_test)
assert train["FLAT_TYPE_ORD"].notna().all() and test["FLAT_TYPE_ORD"].notna().all()

print("flat-type labels before cleaning:", raw_train["FLAT_TYPE"].nunique(), "->", train["FLAT_TYPE"].nunique())
print("streets:", train["STREET"].nunique(), " block + street pairs:", train[["BLOCK", "STREET"]].drop_duplicates().shape[0])
for name, df in [("train", train), ("test", test)]:
    print(f"{name:<5} {len(df):>7,} rows  {df['RENT_APPROVAL_DATE'].min():%Y-%m} to {df['RENT_APPROVAL_DATE'].max():%Y-%m}")

### 2. Block-level location features (slides 5, 12)

Built as in `06_distance_features` §2 from the cleaned tables in `cleaned/` (slide 5: lowercased join key, 11
blocks' `-1` max floor / year completed set to missing, 8 school coordinates corrected) plus the raw mall and
hawker files. Planned MRT stations and hawker centres under construction are excluded. CBD = Raffles Place MRT.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dlat, dlon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


hdb = pd.read_csv(CLEANED / "sg_hdb_block_cleaned.csv", dtype={"POSTAL_CODE": str})
mrt = pd.read_csv(CLEANED / "sg_mrt_stations_cleaned.csv")
schools = pd.read_csv(CLEANED / "sg_schools_cleaned.csv")
malls = lower_strip(pd.read_csv(DATA / "auxiliary" / "sg-shopping-malls.csv"))

with open(DATA / "auxiliary" / "HawkerCentresGEOJSON.geojson") as f:
    features = json.load(f)["features"]
hawker = lower_strip(pd.DataFrame({
    "NAME": [feat["properties"]["NAME"] for feat in features],
    "STATUS": [feat["properties"]["STATUS"] for feat in features],
    "LONGITUDE": [feat["geometry"]["coordinates"][0] for feat in features],
    "LATITUDE": [feat["geometry"]["coordinates"][1] for feat in features],
}))

mrt_open = mrt[mrt["STATUS"].eq("open")].drop_duplicates("NAME")
POI_GROUPS = {
    "primary": schools[schools["MAINLEVEL_CODE"].str.contains("primary", na=False)],
    "secondary": schools[schools["MAINLEVEL_CODE"].str.contains("secondary", na=False)],
    "mrt": mrt_open,
    "mrt_interchange": mrt_open[mrt_open["IS_INTERCHANGE"]],
    "malls": malls,
    "hawker": hawker[hawker["STATUS"].ne("under construction")],
}
RADII_KM = {"1km": 1.0, "2km": 2.0}

lat, lon = hdb["LATITUDE"].to_numpy(), hdb["LONGITUDE"].to_numpy()
for prefix, poi in POI_GROUPS.items():
    d = haversine_km(lat[:, None], lon[:, None], poi["LATITUDE"].to_numpy()[None, :], poi["LONGITUDE"].to_numpy()[None, :])
    hdb[f"{prefix}_nearest_km"] = d.min(axis=1)
    for label, r in RADII_KM.items():
        hdb[f"{prefix}_within_{label}"] = (d <= r).sum(axis=1)
cbd = mrt.loc[mrt["NAME"].eq("raffles place mrt station"), ["LATITUDE", "LONGITUDE"]].iloc[0]
hdb["dist_to_cbd_km"] = haversine_km(lat, lon, cbd["LATITUDE"], cbd["LONGITUDE"])

print("hawker centres:", len(hawker), "->", len(POI_GROUPS["hawker"]), "after dropping those under construction")
print({k: len(v) for k, v in POI_GROUPS.items()})

`DIST` is the stacked set behind slide 12's 496.4 → 489.6: `06_distance_features` §4e, "2km REFERENCE + kitchen
sink". It mixes counts within 1 and 2 km with nearest distances, plus `MAX_FLOOR` and distance to the CBD.

In [ ]:
DIST = [
    "MAX_FLOOR", "primary_within_2km", "mrt_within_2km",
    "primary_nearest_km", "mrt_nearest_km", "secondary_within_1km",
    "mrt_interchange_within_1km", "mrt_interchange_nearest_km",
    "malls_within_1km", "malls_nearest_km", "hawker_within_1km", "hawker_nearest_km", "dist_to_cbd_km",
]

block_cols = KEY + ["MISSING_BUILDING_INFO"] + DIST
train = train.merge(hdb[block_cols], on=KEY, how="left", validate="m:1")
test = test.merge(hdb[block_cols], on=KEY, how="left", validate="m:1")
for name, df in [("train", train), ("test", test)]:
    print(f"{name:<5} rows matched to a block: {df['MISSING_BUILDING_INFO'].notna().mean():.2%}  "
          f"SERS rows (MAX_FLOOR missing): {int(df['MISSING_BUILDING_INFO'].eq(True).sum())}")
train[DIST].describe().T[["mean", "min", "50%", "max"]]

### 3. Validation (slide 8)

Split by time only. Two schemes, rewritten from `validation.py`:

- **Five walk-forward folds** (`expanding_window_splits`): each trains on every month before a cutoff and scores
  the next 3. The last fold scores 2025-01 to 2025-03. The headline is the mean, but folds are compared one by one.
- **One 16-month fold** (`horizon_matched_split`): train through 2023-11, score 2023-12 to 2025-03, the same
  distance ahead as the test window.
- **`horizon_report`** breaks the 16-month fold's error down by months ahead (1-3, 4-6, 7-9, 10-12, 13-16). Bias is
  mean(prediction − actual), so negative is under-prediction.

In [ ]:
def expanding_window_splits(df, n_splits=5, valid_months=3):
    months = df["MONTH_INDEX"].to_numpy()
    unique = np.sort(np.unique(months))
    for k in range(n_splits, 0, -1):
        v_end = len(unique) - (k - 1) * valid_months
        v_start = v_end - valid_months
        yield np.flatnonzero(months < unique[v_start]), np.flatnonzero(np.isin(months, unique[v_start:v_end]))


def horizon_matched_split(df, horizon_months=16):
    months = df["MONTH_INDEX"].to_numpy()
    cutoff = months.max() - horizon_months + 1
    return np.flatnonzero(months < cutoff), np.flatnonzero(months >= cutoff)


def rmse(y, pred):
    return float(np.sqrt(np.mean((np.asarray(y, dtype=float) - np.asarray(pred, dtype=float)) ** 2)))


def bias(y, pred):
    return float(np.mean(np.asarray(pred, dtype=float) - np.asarray(y, dtype=float)))


def horizon_report(df, y_true, y_pred, trn, val, buckets=((1, 3), (4, 6), (7, 9), (10, 12), (13, 16))):
    months = df["MONTH_INDEX"].to_numpy()
    ahead = months[val] - months[trn].max()
    t, p = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    rows = []
    for lo, hi in buckets:
        sel = (ahead >= lo) & (ahead <= hi)
        rows.append({"months ahead": f"{lo}-{hi}", "n": int(sel.sum()), "rmse": rmse(t[sel], p[sel]), "bias": bias(t[sel], p[sel])})
    return pd.DataFrame(rows).set_index("months ahead")


month_label = lambda m: f"{2021 + m // 12}-{m % 12 + 1:02d}"
FOLDS = list(expanding_window_splits(train)) + [horizon_matched_split(train)]
FOLD_NAMES = [f"fold {i}" for i in range(1, 6)] + ["16-month"]
months = train["MONTH_INDEX"].to_numpy()
pd.DataFrame([
    {"fold": name, "train": f"2021-01 to {month_label(months[t].max())}", "scored": f"{month_label(months[v].min())} to {month_label(months[v].max())}",
     "train rows": len(t), "scored rows": len(v)}
    for name, (t, v) in zip(FOLD_NAMES, FOLDS)
]).set_index("fold")

### 4. Baseline and the horizon problem (slide 9)

LightGBM on the six raw columns with the logged settings (`n_estimators=400, learning_rate=0.05, num_leaves=63`,
LightGBM defaults otherwise). Text columns are pandas categories with the level set fixed once from all of train,
as `baselines.model_matrix` does, so train, folds and test share one code per level. Report: five folds 540.4,
517.6, 484.3, 458.2, 466.5 (mean 493.4); 16-month fold 506.2; bias −7 at 1-3 months ahead, −80 at 13-16.

In [ ]:
RAW_FEATURES = ["FLAT_AGE", "FLAT_MODEL", "FLAT_TYPE", "FLOOR_AREA_SQM", "MONTH_INDEX", "TOWN"]
LGBM_PARAMS = {"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 63}
CATEGORIES = {c: pd.CategoricalDtype(sorted(train[c].dropna().unique())) for c in ["FLAT_MODEL", "FLAT_TYPE", "TOWN", "STREET"]}


def model_matrix(df, features):
    # Fixed level sets from train; a level train never had becomes missing.
    X = df[features].copy()
    for c in features:
        if c in CATEGORIES:
            X[c] = X[c].astype(CATEGORIES[c])
    return X


def lgbm(X, y):
    return lgb.LGBMRegressor(**LGBM_PARAMS, verbose=-1).fit(X, y)


def run_folds(predict, label):
    # predict(trn, val, df) -> predictions for df.iloc[val]. Scores the five folds and the 16-month fold.
    y = train[TARGET].to_numpy()
    rows, horizon = [], None
    for name, (t, v) in zip(FOLD_NAMES, FOLDS):
        pred = predict(train, t, v)
        rows.append({"run": label, "fold": name, "rmse": rmse(y[v], pred), "bias": bias(y[v], pred)})
        if name == "16-month":
            horizon = horizon_report(train, y[v], pred, t, v)
    return pd.DataFrame(rows), horizon


def fold_table(results):
    rm = results.pivot(index="run", columns="fold", values="rmse")[FOLD_NAMES]
    rm.insert(5, "5-fold mean", rm[FOLD_NAMES[:5]].mean(axis=1))
    return rm.loc[results["run"].unique()]


def raw_predict(features):
    def predict(df, t, v):
        X, y = model_matrix(df, features), df[TARGET].to_numpy()
        return lgbm(X.iloc[t], y[t]).predict(X.iloc[v])
    return predict


base_results, base_horizon = run_folds(raw_predict(RAW_FEATURES), "baseline (6 raw columns)")
display(fold_table(base_results))
base_results.pivot(index="run", columns="fold", values="bias")[FOLD_NAMES]

In [ ]:
base_horizon

- **Reproduces slide 9 exactly:** folds 540.4, 517.6, 484.3, 458.2, 466.5, mean 493.4; 16-month fold 506.2.
- **Fold spread is 82** (540.4 to 458.2), larger than any feature gain in this notebook, so later runs are
  compared fold by fold.
- **Under-prediction grows with the horizon:** bias −7.2 at 1-3 months ahead, −24.6, −63.2, −67.3, then −80.1 at
  13-16. RMSE falls over the same range (544.7 → 476.7), so RMSE alone hides it. The tree holds the level of its
  last training month.

### 5. Location features (slide 12)

**Reproduction on `06`'s single split** (fit 2021-01 to 2023-12, score 2024-01 to 2025-03), with `06`'s own
columns and settings (`num_leaves=31`, categories from the fit rows): BASE should give ≈ 496.4 and BASE + `DIST`
≈ 489.6.

In [ ]:
def fit_predict_06(num, cat, fit_df, apply_df):
    Xf, Xa = fit_df[num + cat].copy(), apply_df[num + cat].copy()
    for c in cat:
        Xf[c] = Xf[c].astype("category")
        Xa[c] = pd.Categorical(Xa[c].where(Xa[c].isin(Xf[c].cat.categories)), categories=Xf[c].cat.categories)
    est = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, verbose=-1, n_jobs=-1, random_state=SEED)
    return est.fit(Xf, fit_df[TARGET], categorical_feature=cat).predict(Xa)


NUM_06, CAT_06 = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", "MONTH_INDEX"], ["TOWN", "FLAT_TYPE", "FLAT_MODEL"]
cut = pd.Timestamp("2024-01-01")
fit_06, val_06 = train[train["RENT_APPROVAL_DATE"] < cut], train[train["RENT_APPROVAL_DATE"] >= cut]
y_06 = val_06[TARGET].to_numpy()
pd.DataFrame({
    name: {"rmse": rmse(y_06, p), "bias": bias(y_06, p)}
    for name, p in [("BASE", fit_predict_06(NUM_06, CAT_06, fit_06, val_06)),
                    ("BASE + DIST", fit_predict_06(NUM_06 + DIST, CAT_06, fit_06, val_06))]
}).T

**Same features on the slide-8 folds.** The baseline's six raw columns plus all of `DIST`, baseline settings. A
plain result: no per-feature selection, which is next step 2 and not yet done.

In [ ]:
dist_results, dist_horizon = run_folds(raw_predict(RAW_FEATURES + DIST), "baseline + DIST")
loc_results = pd.concat([base_results, dist_results])
display(fold_table(loc_results))
loc_results.pivot(index="run", columns="fold", values="bias").loc[loc_results["run"].unique(), FOLD_NAMES]

- **Reproduces slide 12:** BASE 496.4 → BASE + `DIST` 489.6 (+6.8) on `06`'s single split.
- **On the slide-8 folds, `DIST` helps on every fold:** 540.4 → 533.2, 517.6 → 511.1, 484.3 → 479.6,
  458.2 → 450.6, 466.5 → 452.9 (mean 493.4 → 485.5); 16-month fold 506.2 → 494.9.
- Gains per fold (4.7 to 13.7) are smaller than the spread across folds (82), which is why they are read fold by
  fold. Bias barely moves (16-month −50.0 → −47.4): location does not fix the level.
- This is all 13 `DIST` columns at once. Which ones beat fold noise on their own is next step 2, not done here.

### 6. Market level × flat premium (slide 11)

Following `market_level.py`:

  rent ≈ market level(month) × premium(flat)

- **Level:** a hedonic index, the month coefficients of an OLS of log rent on month dummies plus `TOWN`,
  `FLAT_TYPE`, `FLAT_MODEL` (dummies), `FLOOR_AREA_SQM` and `FLAT_AGE`. Fitted on training months only.
- **Premium:** LightGBM with the baseline settings learns rent ÷ level on the five raw columns other than
  `MONTH_INDEX`; prediction = premium × forecast level.
- **Level forecasts** past the last training month:
  - frozen: the current baseline (section 4), a tree on `MONTH_INDEX` that holds the last month's level;
  - damped trend: Holt's additive trend with damping (`statsmodels`, estimated initial values);
  - straight line: a line fitted through all training months' log level, continued from the last month;
  - perfect level: the level fitted with the scored months' own rents too. Sees the answers; an upper bound.
- **Two cutoffs, each scored on the next 16 months:** plateau (train to 2023-11, the 16-month fold from section
  3) and climb (train to 2022-06, score 2022-07 to 2023-10).

In [ ]:
HEDONIC_CAT, HEDONIC_NUM = ["TOWN", "FLAT_TYPE", "FLAT_MODEL"], ["FLOOR_AREA_SQM", "FLAT_AGE"]


def hedonic_level(df, idx):
    # Log market level per MONTH_INDEX from the rows in idx, relative to the first month.
    rows = df.iloc[idx]
    months = rows["MONTH_INDEX"]
    X = pd.concat([
        pd.get_dummies(months, prefix="m", drop_first=True, dtype=float),
        pd.get_dummies(rows[HEDONIC_CAT], drop_first=True, dtype=float),
        rows[HEDONIC_NUM].astype(float),
    ], axis=1)
    X.insert(0, "const", 1.0)
    coef, *_ = np.linalg.lstsq(X.to_numpy(), np.log(rows[TARGET].to_numpy()), rcond=None)
    coef = pd.Series(coef, index=X.columns)
    ordered = np.sort(months.unique())
    return pd.Series([0.0] + [coef[f"m_{m}"] for m in ordered[1:]], index=ordered)


def forecast_level(level, months, method):
    # Carry a training-months log level forward to the months past its end.
    ahead = np.sort(np.unique(months[months > level.index.max()]))
    if method == "damped":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")  # statsmodels convergence warnings; the fit is unchanged
            fit = ExponentialSmoothing(level.to_numpy(), trend="add", damped_trend=True,
                                       initialization_method="estimated").fit()
        future = fit.forecast(len(ahead))
    elif method == "linear":
        slope, _ = np.polyfit(level.index, level.to_numpy(), 1)
        future = level.iloc[-1] + slope * np.arange(1, len(ahead) + 1)
    elif method == "hold":
        future = np.full(len(ahead), level.iloc[-1])
    else:
        raise ValueError(method)
    return pd.concat([level, pd.Series(future, index=ahead)])


def level_scale(df, t, v, method):
    # Market level (not log) for every row of df; v rows get the forecast.
    months = df["MONTH_INDEX"].to_numpy()
    if method == "oracle":
        level = hedonic_level(df, np.concatenate([t, v]))
    else:
        level = forecast_level(hedonic_level(df, t), months[v], method)
    return np.exp(level.reindex(months).to_numpy())


def level_predict(features, method):
    def predict(df, t, v):
        X, y = model_matrix(df, features), df[TARGET].to_numpy()
        scale = level_scale(df, t, v, method)
        return lgbm(X.iloc[t], y[t] / scale[t]).predict(X.iloc[v]) * scale[v]
    return predict


PREMIUM_FEATURES = [c for c in RAW_FEATURES if c != "MONTH_INDEX"]
ARMS = {
    "frozen (current baseline)": raw_predict(RAW_FEATURES),
    "damped trend": level_predict(PREMIUM_FEATURES, "damped"),
    "straight line": level_predict(PREMIUM_FEATURES, "linear"),
    "perfect level (upper bound)": level_predict(PREMIUM_FEATURES, "oracle"),
}
CUTOFFS = {"plateau (to 2023-11)": train, "climb (to 2022-06)": train[train["MONTH_INDEX"] <= 33].reset_index(drop=True)}

level_rows = []
for cutoff, df in CUTOFFS.items():
    t, v = horizon_matched_split(df)
    y = df[TARGET].to_numpy()
    for arm, predict in ARMS.items():
        pred = predict(df, t, v)
        h = horizon_report(df, y[v], pred, t, v)
        level_rows.append({"forecast": arm, "cutoff": cutoff, "rmse": rmse(y[v], pred), "bias 13-16": h.loc["13-16", "bias"]})
level_results = pd.DataFrame(level_rows)
level_results.pivot(index="forecast", columns="cutoff", values=["rmse", "bias 13-16"]).loc[list(ARMS)]

- **Reproduces slide 11 exactly:** frozen 506.2 / 793.3, damped 500.1 / 620.0, straight line 656.2 / 661.9,
  perfect level 499.9 / 611.0 (plateau / climb).
- **Damped trend is close to the upper bound at both cutoffs:** 0.2 above it on the plateau, 9.0 above it in the
  climb. It cuts 13-16-month bias from −80.1 to −49.9 on the plateau and from −619.7 to +86.5 in the climb.
- **The straight line overshoots on the plateau** (+651.8 bias at 13-16 months) because its slope averages in the
  2021-23 climb.

**Caveat:** a wider check after the report (nine cutoffs, see `08_level_robustness.ipynb`) found the damped trend
over-predicts badly from a 2022-12 cutoff (bias about +533, RMSE about 871), so this two-cutoff result is not
conclusive.

### 7. Macro data (slide 10)

Not used. On the `macro-time-trend` branch (`src/hdb_rentals/macro.py`, `experiments.jsonl`), COE prices did no
better than an unrelated share price used as a placebo (504.4 vs 505.6 vs 506.2 on the 16-month fold), so macro
features were dropped. Not re-run here.

### 8. Combined model (next step 3)

Location features in the premium model, damped-trend level on top:

- **Premium columns** are slide 4's: `FLAT_TYPE_ORD` (1 to 6), `FLAT_AGE`, `TOWN`, `FLAT_MODEL`, `FLOOR_AREA_SQM`
  (`main.ipynb` keeps it "for now"), `STREET`, no `BLOCK`, plus all of `DIST`. `main.ipynb` one-hot encodes
  `STREET`; here it is a LightGBM native categorical (576 levels), with the level set fixed from all of train.
  No `MONTH_INDEX`: time enters only through the level.
- **Level:** hedonic index, damped-trend forecast, as in section 6.
- Baseline settings for LightGBM. Scored on the slide-8 folds next to the baseline, with the damped level on
  the raw columns (section 6's model) as the step in between.

In [ ]:
SLIDE4_FEATURES = ["FLAT_TYPE_ORD", "FLAT_AGE", "TOWN", "FLAT_MODEL", "FLOOR_AREA_SQM", "STREET"]
COMBINED_FEATURES = SLIDE4_FEATURES + DIST
LEVEL_METHOD = "damped"

damped_raw_results, _ = run_folds(level_predict(PREMIUM_FEATURES, LEVEL_METHOD), "damped level x raw premium")
comb_results, combined_horizon = run_folds(level_predict(COMBINED_FEATURES, LEVEL_METHOD),
                                           "combined: damped level x (slide-4 + DIST) premium")
combined_results = pd.concat([base_results, damped_raw_results, comb_results])
display(fold_table(combined_results))
combined_results.pivot(index="run", columns="fold", values="bias").loc[combined_results["run"].unique(), FOLD_NAMES]

In [ ]:
pd.concat({"baseline": base_horizon, "combined": combined_horizon}, axis=1)

- **`STREET` as a native categorical works:** no errors, no unseen streets in any fold or in test, and it does
  not make any fold worse, so no fallback was needed.
- **The combined model beats the baseline on every fold:** 540.4 → 528.3, 517.6 → 507.6, 484.3 → 476.5,
  458.2 → 445.4, 466.5 → 450.6 (mean 493.4 → 481.7, −11.7); 16-month fold 506.2 → 488.9 (−17.3).
- **Each part helps on its own.** The damped level on the raw columns gives mean 491.2 and 16-month 500.1 (both
  logged on `macro-time-trend`). Adding the slide-4 columns and `DIST` to the premium takes that to 481.7 and
  488.9. `DIST` with `MONTH_INDEX` and no level (section 5) gives 485.5 and 494.9.
- **Horizon bias halves** at every distance past 6 months: −80.1 → −43.7 at 13-16 months, −63.2 → −32.1 at 7-9.
  The level is still under-forecast over 16 months on this cutoff.
- **On the short folds, the level now leans high:** bias +28.5 and +31.1 on folds 1 and 2 (baseline −2.0, −2.9).
  The damped trend still carries some of the 2021-23 climb when trained to 2023-12 or 2024-03.
- One configuration, baseline settings, no tuning. Columns and `DIST` were not selected per feature.

### 9. Fit on all of train, first submission (next step 4)

The hedonic level is fitted on 2021-01 to 2025-03 and forecast with the damped trend for 2025-04 to 2026-07
(test rows from 2025-03 use that month's fitted level). The combined premium model is trained on all of train
and predicts the 50,000 test rows in the file's own order. The test file has no ID column, so the submission is
a single `MONTHLY_RENT` column matched by row position.

In [ ]:
full = pd.concat([train, test.assign(**{TARGET: np.nan})], ignore_index=True)
t_all, v_all = np.arange(len(train)), np.arange(len(train), len(full))
scale = level_scale(full, t_all, v_all, LEVEL_METHOD)
X = model_matrix(full, COMBINED_FEATURES)
y_train = train[TARGET].to_numpy()
final = lgbm(X.iloc[t_all], y_train / scale[t_all])
test_pred = final.predict(X.iloc[v_all]) * scale[v_all]

unseen_streets = test["STREET"][~test["STREET"].isin(CATEGORIES["STREET"].categories)]
print(f"test rows on a street train never had: {len(unseen_streets)} (predicted with STREET missing)")
assert len(test_pred) == len(raw_test) and np.isfinite(test_pred).all()

submission = pd.DataFrame({TARGET: test_pred})
submission.to_csv(OUT / "submission.csv", index=False)
print(f"wrote {OUT / 'submission.csv'}: {len(submission):,} rows")

test_month = test["RENT_APPROVAL_DATE"].dt.strftime("%Y-%m").to_numpy()
level_by_month = pd.Series(scale[v_all], index=test_month).groupby(level=0).first()
summary = pd.DataFrame({"level (2021-01 = 1)": level_by_month,
                        "median prediction": pd.Series(test_pred, index=test_month).groupby(level=0).median()})
print(f"level change, 2025-03 to 2026-07: {level_by_month.iloc[-1] / level_by_month.iloc[0] - 1:+.1%}")
print("actual median rent, 2025-01 to 2025-03:",
      train[train["MONTH_INDEX"] >= 48].groupby("MONTH_INDEX")[TARGET].median().to_dict())
summary.iloc[[0, 4, 8, 12, 16]]

- `outputs/submission.csv`: 50,000 rows, one `MONTHLY_RENT` column, test file order. Every test street was seen
  in train.
- **The damped trend raises the level 4.3% over the test window** (1.56 at 2025-03 to 1.63 at 2026-07, 2021-01 =
  1). Median prediction goes from 3,180 to 3,321. The last training months' actual medians are 3,100, 3,150 and
  3,200.
- Not yet compared with the leaderboard; that is the second half of next step 4.

### 10. Summary

**Reproduced from the progress report, all within 0.1 of the slides:**

- Slide 8/9 baseline: five folds 540.4, 517.6, 484.3, 458.2, 466.5 (mean 493.4); 16-month fold 506.2; bias −7.2 at
  1-3 months ahead, −80.1 at 13-16.
- Slide 11 level forecasts (plateau / climb): frozen 506.2 / 793.3, damped 500.1 / 620.0, straight line
  656.2 / 661.9, perfect 499.9 / 611.0. Caveat: a nine-cutoff check after the report
  (`08_level_robustness.ipynb`) shows the damped trend can overshoot badly (+533 bias from 2022-12).
- Slide 12 location features: 496.4 → 489.6 on `06`'s split. On the slide-8 folds, mean 493.4 → 485.5 and
  16-month 506.2 → 494.9.
- Slide 10 macro: not used, not re-run.

**Next steps covered here:**

- **3. Combine:** damped level × LightGBM premium on slide 4's columns (`STREET` as a native categorical) +
  `DIST`. Five folds 528.3, 507.6, 476.5, 445.4, 450.6 (mean 481.7 vs 493.4); 16-month 488.9 vs 506.2. Better on
  every fold.
- **4. First submission:** `outputs/submission.csv` from the combined model. Leaderboard comparison still to do.

**Still open:**

- **1.** Adopt the five folds + 16-month fold as the only scheme and log every run to `experiments.jsonl`. This
  notebook uses the scheme but does not log.
- **2.** Re-score location features one by one on these folds; keep only those that beat fold noise. Here all 13
  `DIST` columns go in together.
- **5.** Other models and tuning on the premium part.

In [ ]:
print(f"runtime: {(time.time() - T0) / 60:.1f} min")

## Part 2.2. HDB Rentals: model training on the progress report's setup

The progress report fixed the features and the time treatment. This notebook runs its next steps (slide 12) as
**three experiments, one diagnostic and one submission**. Every experiment changes one thing on the same setup. Each
section opens with a card (question, what is varied, what is held fixed, which split chooses) and closes with
its findings and a **Decision** line.

### Experiment log

| ID | Experiment | Varied (everything else fixed) | Chosen on | Holdout RMSE | Outcome |
|---|---|---|---|---|---|
| **E1** | Model families, default settings | Library: linear regression, random forest, HistGradientBoosting, LightGBM, XGBoost, CatBoost | nothing tuned; holdout only shortlists | LightGBM **489.6** (= slide 11), HGB 490.7, XGBoost 496.7, CatBoost 498.4, RF 503.3, LR 652.6 | LightGBM, XGBoost, CatBoost go on to E2 |
| **E2** | Hyperparameter search | Tree shape and regularisation: starting point + 20 random draws per booster; rounds by early stopping | tuning split | LightGBM **487.5** (−2.1), XGBoost 489.6 (−7.0), CatBoost 490.4 (−8.1) | Tuned LightGBM is the best single model |
| **E3** | Blending | Weighted average of E2's three tuned boosters | tuning split | 488.0 (equal weights 487.9) | No gain: the boosters' errors correlate at 0.99 |
| D1 | Error by months ahead | Nothing is fit; E1's LightGBM and E3's blend broken down by horizon | – | bias about 0 at 1-3 months ahead, −64 at 13-15 | The error is the market level, not the model |
| S1 | Kaggle dry run | E2's boosters and E3's blend refit on all of train | – | – | `submission_09_lgbm.csv` (public leaderboard 443.25, recorded in `10`), `submission_09_blend.csv` |

### Held fixed in every experiment (decided in the report)

| decision | slide |
|---|---|
| Clean as slide 5: lowercase text, "4 room" = "4-room", drop `FURNISHED` / `FEE`, month counter `MONTH_INDEX` | 5 |
| Join HDB block table on town + block + street; planned MRTs and hawker centres under construction excluded | 6, 9 |
| Time enters only through `MONTH_INDEX`. No COE or other macro series, no month-of-year seasonality | 8 |
| Features: BASE (`FLOOR_AREA_SQM`, `LEASE_COMMENCE_DATE`, `MONTH_INDEX`, `TOWN`, `FLAT_TYPE`, `FLAT_MODEL`) + the distance set behind LightGBM 489.6 | 11 |
| Evaluation holdout: fit 2021-01 to 2023-12, score 2024-01 to 2025-03 | 11 |

The market-level work in `07_pipeline.ipynb` / `08_level_robustness.ipynb` came after the report and is not
used here.

**Contents.** Setup: 1 Cleaning · 2 Location features · 3 Evaluation protocol. Experiments: E1 · E2 · E3.
Diagnostic: D1. Submission: S1. Summary.

In [ ]:
import json
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams["figure.dpi"] = 110
warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA, CLEANED, OUT = ROOT / "data", ROOT / "cleaned", ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
TARGET = "MONTHLY_RENT"
KEY = ["TOWN", "BLOCK", "STREET"]
SEED = 0
T0 = time.time()

### 1. Setup: cleaning (slide 5)

Same cleaning as `07` §1. Train is sorted by month; test keeps the file's order, since the submission is matched
by row position.

In [ ]:
def lower_strip(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    text = [c for c in out.columns if pd.api.types.is_string_dtype(out[c])]
    out[text] = out[text].apply(lambda s: s.str.strip().str.lower())
    return out


def clean_rent(df: pd.DataFrame) -> pd.DataFrame:
    out = lower_strip(df).drop(columns=["FURNISHED", "FEE"])
    # "4 room" and "4-room" are the same level.
    out["FLAT_TYPE"] = out["FLAT_TYPE"].str.replace(r"[\s\-]+", "-", regex=True)
    date = pd.to_datetime(out["RENT_APPROVAL_DATE"], format="%Y-%m")
    out["RENT_APPROVAL_DATE"] = date
    out["MONTH_INDEX"] = (date.dt.year - 2021) * 12 + date.dt.month - 1
    return out


raw_train, raw_test = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
train = clean_rent(raw_train).sort_values("MONTH_INDEX", kind="stable").reset_index(drop=True)
test = clean_rent(raw_test)
for name, df in [("train", train), ("test", test)]:
    print(f"{name:<5} {len(df):>7,} rows  {df['RENT_APPROVAL_DATE'].min():%Y-%m} to {df['RENT_APPROVAL_DATE'].max():%Y-%m}")

### 2. Setup: location features (slides 6, 11)

Built as in `06_distance_features` §2 and `07` §2: haversine distances from every HDB block to open MRT stations,
primary and secondary schools, malls and open hawker centres, counted within 1 and 2 km and to the nearest one,
plus distance to the CBD (Raffles Place MRT).

`DIST` is the 13-column set behind slide 11's LightGBM 489.6 ("REFERENCE + everything new"): the 2 km school
and MRT counts, `MAX_FLOOR`, the 1 km mall and hawker counts, and the nearest distances.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dlat, dlon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


hdb = pd.read_csv(CLEANED / "sg_hdb_block_cleaned.csv", dtype={"POSTAL_CODE": str})
mrt = pd.read_csv(CLEANED / "sg_mrt_stations_cleaned.csv")
schools = pd.read_csv(CLEANED / "sg_schools_cleaned.csv")
malls = lower_strip(pd.read_csv(DATA / "auxiliary" / "sg-shopping-malls.csv"))

with open(DATA / "auxiliary" / "HawkerCentresGEOJSON.geojson") as f:
    features = json.load(f)["features"]
hawker = lower_strip(pd.DataFrame({
    "NAME": [feat["properties"]["NAME"] for feat in features],
    "STATUS": [feat["properties"]["STATUS"] for feat in features],
    "LONGITUDE": [feat["geometry"]["coordinates"][0] for feat in features],
    "LATITUDE": [feat["geometry"]["coordinates"][1] for feat in features],
}))

mrt_open = mrt[mrt["STATUS"].eq("open")].drop_duplicates("NAME")
POI_GROUPS = {
    "primary": schools[schools["MAINLEVEL_CODE"].str.contains("primary", na=False)],
    "secondary": schools[schools["MAINLEVEL_CODE"].str.contains("secondary", na=False)],
    "mrt": mrt_open,
    "mrt_interchange": mrt_open[mrt_open["IS_INTERCHANGE"]],
    "malls": malls,
    "hawker": hawker[hawker["STATUS"].ne("under construction")],
}
RADII_KM = {"1km": 1.0, "2km": 2.0}

lat, lon = hdb["LATITUDE"].to_numpy(), hdb["LONGITUDE"].to_numpy()
for prefix, poi in POI_GROUPS.items():
    d = haversine_km(lat[:, None], lon[:, None], poi["LATITUDE"].to_numpy()[None, :], poi["LONGITUDE"].to_numpy()[None, :])
    hdb[f"{prefix}_nearest_km"] = d.min(axis=1)
    for label, r in RADII_KM.items():
        hdb[f"{prefix}_within_{label}"] = (d <= r).sum(axis=1)
cbd = mrt.loc[mrt["NAME"].eq("raffles place mrt station"), ["LATITUDE", "LONGITUDE"]].iloc[0]
hdb["dist_to_cbd_km"] = haversine_km(lat, lon, cbd["LATITUDE"], cbd["LONGITUDE"])

DIST = [
    "MAX_FLOOR", "primary_within_2km", "mrt_within_2km",
    "primary_nearest_km", "mrt_nearest_km", "secondary_within_1km",
    "mrt_interchange_within_1km", "mrt_interchange_nearest_km",
    "malls_within_1km", "malls_nearest_km", "hawker_within_1km", "hawker_nearest_km", "dist_to_cbd_km",
]
train = train.merge(hdb[KEY + DIST], on=KEY, how="left", validate="m:1")
test = test.merge(hdb[KEY + DIST], on=KEY, how="left", validate="m:1")
assert len(train) == len(raw_train) and len(test) == len(raw_test)
print("rows with no block match: train", int(train["dist_to_cbd_km"].isna().sum()), " test", int(test["dist_to_cbd_km"].isna().sum()))
print("MAX_FLOOR missing (SERS blocks): train", int(train["MAX_FLOOR"].isna().sum()), " test", int(test["MAX_FLOOR"].isna().sum()))

### 3. Setup: evaluation protocol (next step 1)

Two time splits. Every experiment card says which one chooses and which one only reports.

- **Holdout:** fit 2021-01 to 2023-12, score 2024-01 to 2025-03 (15 months), the split behind slide 11. The test
  window is 2025-03 to 2026-07, 1 to 16 months past the last label, so the holdout is scored at a similar
  distance ahead.
- **Tuning split, inside the holdout's fit months:** fit 2021-01 to 2023-03, score 2023-04 to 2023-12. Hyperparameter
  search, early stopping and blend weights are chosen here only, so the holdout is scored once per final
  configuration and is not used to pick settings.

Metrics: RMSE (the competition metric), MAE, and bias = mean(prediction − actual), so negative means
under-prediction.

In [ ]:
HOLDOUT_START, TUNE_START = "2024-01", "2023-04"
date = train["RENT_APPROVAL_DATE"]
fit = train[date < HOLDOUT_START].reset_index(drop=True)
holdout = train[date >= HOLDOUT_START].reset_index(drop=True)
tune_fit = fit[fit["RENT_APPROVAL_DATE"] < TUNE_START].reset_index(drop=True)
tune_val = fit[fit["RENT_APPROVAL_DATE"] >= TUNE_START].reset_index(drop=True)

span = lambda df: f"{df['RENT_APPROVAL_DATE'].min():%Y-%m} to {df['RENT_APPROVAL_DATE'].max():%Y-%m}"
pd.DataFrame([
    {"split": "holdout", "fit": span(fit), "fit rows": len(fit), "scored": span(holdout), "scored rows": len(holdout)},
    {"split": "tuning", "fit": span(tune_fit), "fit rows": len(tune_fit), "scored": span(tune_val), "scored rows": len(tune_val)},
]).set_index("split")

### E1. Model families at default settings (next step 3)

| | |
|---|---|
| **Question** | With the report's features fixed, does another model family beat the report's LightGBM? |
| **Reference** | Report's LightGBM (slide 11: holdout 489.6), which this experiment should reproduce |
| **Varied** | Library: linear regression, random forest, HistGradientBoosting, LightGBM, XGBoost, CatBoost |
| **Held fixed** | Features (`NUM` + `CAT` below), holdout split, each library's usual starting settings. LightGBM uses the report's (`n_estimators=400`, `learning_rate=0.05`, `num_leaves=31`) |
| **Scored on** | Holdout, once per model. No settings are tuned here |
| **Used for** | Which libraries go on to E2 |

**How each model sees the data.** Every model gets the same columns. Categorical level sets are fixed once from
all of train, so the splits and the test file share one code per level. Encoding differs by model:

- **Linear regression:** median-impute and scale numerics, one-hot categoricals (as `03`).
- **Random forest, HistGradientBoosting:** ordinal-coded categoricals (native categorical splits for HGB).
- **LightGBM, XGBoost, CatBoost:** native categoricals, `MAX_FLOOR` left missing for the SERS blocks.

The first cell defines the shared model code (also used by E2, E3 and S1); the second runs E1.

In [ ]:
NUM = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", "MONTH_INDEX"] + DIST
CAT = ["TOWN", "FLAT_TYPE", "FLAT_MODEL"]
FEATURES = NUM + CAT
CATEGORIES = {c: pd.CategoricalDtype(sorted(train[c].unique())) for c in CAT}
assert all(test[c].isin(CATEGORIES[c].categories).all() for c in CAT), "test has a level train never had"


def matrix(df):
    X = df[FEATURES].copy()
    for c in CAT:
        X[c] = X[c].astype(CATEGORIES[c])
    return X


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(p, float)) ** 2)))


def score(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"rmse": rmse(y, p), "mae": float(np.mean(np.abs(p - y))), "bias": float(np.mean(p - y))}


def ordinal_pre():
    cats = [list(CATEGORIES[c].categories) for c in CAT]
    return ColumnTransformer([("cat", OrdinalEncoder(categories=cats), CAT), ("num", "passthrough", NUM)])


def make_model(name, params=None):
    p = params or {}
    if name == "Linear regression":
        pre = ColumnTransformer([
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
        ])
        return Pipeline([("pre", pre), ("m", LinearRegression())])
    if name == "Random forest":
        rf = RandomForestRegressor(**{"n_estimators": 300, "min_samples_leaf": 5, "max_features": 0.5, **p},
                                   n_jobs=-1, random_state=SEED)
        return Pipeline([("pre", ordinal_pre()), ("m", rf)])
    if name == "HistGradientBoosting":
        hgb = HistGradientBoostingRegressor(**{"max_iter": 400, "learning_rate": 0.05, **p},
                                            categorical_features=list(range(len(CAT))), random_state=SEED)
        return Pipeline([("pre", ordinal_pre()), ("m", hgb)])
    if name == "LightGBM":
        return lgb.LGBMRegressor(**{"n_estimators": 400, "learning_rate": 0.05, "num_leaves": 31, **p},
                                 verbose=-1, n_jobs=-1, random_state=SEED)
    if name == "XGBoost":
        return xgb.XGBRegressor(**{"n_estimators": 400, "learning_rate": 0.05, "max_depth": 8, **p},
                                tree_method="hist", enable_categorical=True, max_cat_to_onehot=1,
                                n_jobs=-1, random_state=SEED)
    if name == "CatBoost":
        return CatBoostRegressor(**{"iterations": 1000, "learning_rate": 0.1, "depth": 8, **p},
                                 cat_features=CAT, loss_function="RMSE", thread_count=-1,
                                 random_seed=SEED, verbose=0, allow_writing_files=False)
    raise ValueError(name)


def to_input(name, df):
    # CatBoost wants plain strings for categoricals; the others take the fixed-level frame.
    X = matrix(df)
    if name == "CatBoost":
        X[CAT] = X[CAT].astype(str)
    return X


def fit_predict(name, fit_df, apply_df, params=None):
    est = make_model(name, params)
    est.fit(to_input(name, fit_df), fit_df[TARGET].to_numpy())
    return est.predict(to_input(name, apply_df))

In [ ]:
MODELS = ["Linear regression", "Random forest", "HistGradientBoosting", "LightGBM", "XGBoost", "CatBoost"]
y_hold = holdout[TARGET].to_numpy()

default_rows, hold_preds = [], {}
for name in MODELS:
    t = time.time()
    hold_preds[f"{name} (default)"] = p = fit_predict(name, fit, holdout)
    default_rows.append({"model": name, **score(y_hold, p), "fit + predict (s)": time.time() - t})
default_table = pd.DataFrame(default_rows).set_index("model").sort_values("rmse")
default_table

**E1 findings**

- **LightGBM reproduces slide 11 exactly: 489.58.** Linear regression gives 652.6 (slide 11: 652.4; the spec
  here carries all 13 `DIST` columns).
- **HistGradientBoosting ties LightGBM** (490.7, +1.1). At their starting settings XGBoost (+7.1), CatBoost
  (+8.9) and random forest (+13.7) are behind.
- **Every tree under-predicts by 34 to 40**; linear regression over-predicts by 391. A linear `MONTH_INDEX`
  keeps extending the 2021-23 climb into the 2024 plateau; a tree holds the last level (slide 8).

**Decision:** LightGBM, XGBoost and CatBoost go on to E2. Linear regression and random forest are dropped (well
behind); HistGradientBoosting is not tuned, since it is LightGBM's algorithm and E2 covers LightGBM.

### E2. Tuning the boosters (next step 3)

| | |
|---|---|
| **Question** | How much do the boosters gain from tuning tree shape and regularisation? |
| **Reference** | E1's default-settings holdout scores |
| **Varied** | Hyperparameters of LightGBM, XGBoost and CatBoost: E1's starting point + 20 random draws each (search space in `SPACE` below) |
| **Held fixed** | Features, learning rate 0.05. Rounds are found by early stopping (patience 100, at most 5,000), not chosen |
| **Chosen on** | Tuning split: the draw with the lowest tuning RMSE per library |
| **Scored on** | Holdout, once per library: the best draw is refit on the holdout's fit months and scored |
| **Not tuned** | Linear regression, random forest, HistGradientBoosting (see E1's decision) |

In [ ]:
N_DRAWS = 20
LR = 0.05
MAX_ROUNDS = 5000
rng = np.random.default_rng(SEED)
y_tune = tune_val[TARGET].to_numpy()

SPACE = {
    "LightGBM": (
        {"num_leaves": 31},
        lambda: {"num_leaves": int(rng.choice([31, 63, 127, 255, 511])),
                 "min_child_samples": int(rng.choice([5, 10, 20, 50, 100])),
                 "subsample": float(rng.choice([0.7, 0.85, 1.0])), "subsample_freq": 1,
                 "colsample_bytree": float(rng.choice([0.5, 0.7, 0.85, 1.0])),
                 "reg_lambda": float(rng.choice([0.0, 1.0, 5.0, 20.0])),
                 "cat_smooth": float(rng.choice([1.0, 10.0, 30.0]))},
    ),
    "XGBoost": (
        {"max_depth": 8},
        lambda: {"max_depth": int(rng.choice([6, 8, 10, 12])),
                 "min_child_weight": float(rng.choice([1, 5, 10, 30])),
                 "subsample": float(rng.choice([0.7, 0.85, 1.0])),
                 "colsample_bytree": float(rng.choice([0.5, 0.7, 0.85, 1.0])),
                 "reg_lambda": float(rng.choice([1.0, 5.0, 20.0])),
                 "max_cat_threshold": int(rng.choice([16, 32, 64]))},
    ),
    "CatBoost": (
        {"depth": 8},
        lambda: {"depth": int(rng.choice([6, 8, 10])),
                 "l2_leaf_reg": float(rng.choice([1.0, 3.0, 10.0, 30.0])),
                 "random_strength": float(rng.choice([0.5, 1.0, 2.0])),
                 "rsm": float(rng.choice([0.5, 0.75, 1.0])),
                 "one_hot_max_size": int(rng.choice([2, 10, 50]))},
    ),
}


def fit_early_stopped(name, params, fit_df, eval_df):
    # Train with early stopping on eval_df; return (predictions on eval_df, rounds used).
    Xf, Xe = to_input(name, fit_df), to_input(name, eval_df)
    yf, ye = fit_df[TARGET].to_numpy(), eval_df[TARGET].to_numpy()
    if name == "LightGBM":
        est = make_model(name, {**params, "n_estimators": MAX_ROUNDS, "learning_rate": LR})
        est.fit(Xf, yf, eval_X=Xe, eval_y=ye, callbacks=[lgb.early_stopping(100, verbose=False)])
        return est.predict(Xe), est.best_iteration_
    if name == "XGBoost":
        est = make_model(name, {**params, "n_estimators": MAX_ROUNDS, "learning_rate": LR, "early_stopping_rounds": 100})
        est.fit(Xf, yf, eval_set=[(Xe, ye)], verbose=False)
        return est.predict(Xe, iteration_range=(0, est.best_iteration + 1)), est.best_iteration + 1
    if name == "CatBoost":
        est = make_model(name, {**params, "iterations": MAX_ROUNDS, "learning_rate": LR,
                                "od_type": "Iter", "od_wait": 100, "use_best_model": True})
        est.fit(Xf, yf, eval_set=(Xe, ye))
        return est.predict(Xe), est.get_best_iteration() + 1
    raise ValueError(name)


search_rows, tune_preds = [], {}
for name, (start, draw) in SPACE.items():
    for i, params in enumerate([start] + [draw() for _ in range(N_DRAWS)]):
        t = time.time()
        p, rounds = fit_early_stopped(name, params, tune_fit, tune_val)
        tune_preds[(name, i)] = p
        search_rows.append({"model": name, "draw": i, "tune rmse": rmse(y_tune, p), "rounds": rounds,
                            "seconds": time.time() - t, "params": params})
search = pd.DataFrame(search_rows)
print(f"search time: {search['seconds'].sum() / 60:.1f} min")
search.sort_values("tune rmse").groupby("model").head(3).sort_values(["model", "tune rmse"])[
    ["model", "draw", "tune rmse", "rounds", "params"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for name in SPACE:
    s = search[search["model"] == name]["tune rmse"].sort_values().to_numpy()
    ax.plot(np.arange(1, len(s) + 1), s, marker="o", ms=3, label=name)
ax.set_xlabel("draws, best first")
ax.set_ylabel("RMSE, tuning split (SGD)")
ax.set_title("Random search: spread of tuning-split RMSE per library", loc="left")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Holdout score of each library's best draw.** Refit on 2021-01 to 2023-12 with the chosen settings and number
of rounds, scored once on 2024-01 to 2025-03. The holdout's fit months are 9 more than the tuning split's, so
the round count is scaled up by the same ratio of rows.

In [ ]:
best = search.loc[search.groupby("model")["tune rmse"].idxmin()].set_index("model")
round_scale = len(fit) / len(tune_fit)
ROUNDS_KEY = {"LightGBM": "n_estimators", "XGBoost": "n_estimators", "CatBoost": "iterations"}


def final_params(name, rounds):
    return {**best.loc[name, "params"], ROUNDS_KEY[name]: int(round(rounds * round_scale)), "learning_rate": LR}


tuned_rows = []
for name in SPACE:
    params = final_params(name, best.loc[name, "rounds"])
    t = time.time()
    hold_preds[f"{name} (tuned)"] = p = fit_predict(name, fit, holdout, params)
    tuned_rows.append({"model": name, "default rmse": default_table.loc[name, "rmse"], "tuned rmse": score(y_hold, p)["rmse"],
                       "tuned bias": score(y_hold, p)["bias"], "tune-split rmse": best.loc[name, "tune rmse"],
                       "rounds": params[ROUNDS_KEY[name]], "seconds": time.time() - t})
tuned_table = pd.DataFrame(tuned_rows).set_index("model")
tuned_table["gain"] = tuned_table["default rmse"] - tuned_table["tuned rmse"]
display(tuned_table.sort_values("tuned rmse"))
for name in SPACE:
    print(f"{name:<9}", final_params(name, best.loc[name, "rounds"]))

**E2 findings**

- **Tuning helps, but only by a few dollars:** LightGBM 489.6 → 487.5 (−2.1), XGBoost 496.7 → 489.6 (−7.0),
  CatBoost 498.4 → 490.4 (−8.1). The three libraries end up within 3 of each other.
- **All three searches pick the same kind of model:** shallower, heavily regularised trees. LightGBM
  63 leaves, `min_child_samples=100`, `reg_lambda=20`, half the columns per tree; XGBoost depth 6,
  `min_child_weight=30`, `reg_lambda=20`; CatBoost depth 6, `l2_leaf_reg=30`. Months ahead are a different market
  from the fit months, so smoother fits travel better.
- **The tuning split cannot rank the libraries.** CatBoost is best there (602.4 vs 603.1 and 603.2) and worst on
  the holdout. Differences under 1 on one split are noise. The tuning split's RMSE is higher in absolute terms
  because 2023 rents were still rising, so the held level is further off.

**Decision:** each library's best draw goes on to E3. Tuned LightGBM (487.5) is the best single model; its
settings are the base model of `10`.

### E3. Blending the tuned boosters

| | |
|---|---|
| **Question** | Do the three tuned boosters make different enough errors that averaging them helps? |
| **Reference** | Tuned LightGBM alone (E2: 487.5) |
| **Varied** | Blend weights: a grid over weights that sum to 1, step 0.05. An equal-weight average is shown alongside |
| **Held fixed** | E2's three tuned models and their predictions |
| **Chosen on** | Tuning split (the weights) |
| **Scored on** | Holdout, once, with the weights unchanged |

In [ ]:
BOOSTERS = list(SPACE)
tune_best = np.column_stack([tune_preds[(m, int(best.loc[m, "draw"]))] for m in BOOSTERS])
hold_best = np.column_stack([hold_preds[f"{m} (tuned)"] for m in BOOSTERS])

grid = np.round(np.arange(0, 1.0001, 0.05), 2)
weights = [(a, b, round(1 - a - b, 2)) for a in grid for b in grid if a + b <= 1 + 1e-9]
blend_search = pd.DataFrame([{"w": w, "tune rmse": rmse(y_tune, tune_best @ np.array(w))} for w in weights])
W = np.array(blend_search.loc[blend_search["tune rmse"].idxmin(), "w"])
print("blend weights chosen on the tuning split:", dict(zip(BOOSTERS, W)))

hold_preds["blend (tuned weights)"] = hold_best @ W
hold_preds["blend (equal weights)"] = hold_best.mean(axis=1)

ranked = ["LightGBM (default)"] + [f"{m} (tuned)" for m in BOOSTERS] + ["blend (equal weights)", "blend (tuned weights)"]
blend_table = pd.DataFrame({k: score(y_hold, hold_preds[k]) for k in ranked}).T
blend_table["vs report (489.6)"] = blend_table["rmse"] - default_table.loc["LightGBM", "rmse"]
blend_table

In [ ]:
print("correlation of holdout errors between the tuned boosters:")
pd.DataFrame(hold_best - y_hold[:, None], columns=BOOSTERS).corr()

**E3 findings**

- **Blending buys nothing.** The three boosters' holdout errors correlate at 0.99: they make the same mistakes.
  Equal weights give 487.9, tuning-split weights 488.0; tuned LightGBM alone is 487.5.
- **From the report's LightGBM to the best model here is 2.1 RMSE (0.4%).** The model is not where the error is.

**Decision:** the blend is not better than tuned LightGBM alone. Both are written in S1, so the leaderboard can
say whether it agrees.

### D1. Error by months ahead (diagnostic)

| | |
|---|---|
| **Question** | Do tuning and blending change slide 8's horizon problem? |
| **Compared** | The report's LightGBM (E1) and the tuning-weight blend (E3), on the holdout |
| **Method** | Nothing is fit. Existing holdout predictions are broken down by months past 2023-12, in 3-month buckets |

Background: slide 8 found that a tree on `MONTH_INDEX` holds the level of its last training month, so it
under-predicts more the further ahead it scores.

In [ ]:
ahead = holdout["MONTH_INDEX"].to_numpy() - fit["MONTH_INDEX"].max()
BUCKETS = [(1, 3), (4, 6), (7, 9), (10, 12), (13, 15)]
horizon = {}
for k in ["LightGBM (default)", "blend (tuned weights)"]:
    rows = []
    for lo, hi in BUCKETS:
        sel = (ahead >= lo) & (ahead <= hi)
        rows.append({"months ahead": f"{lo}-{hi}", "n": int(sel.sum()), **score(y_hold[sel], hold_preds[k][sel])})
    horizon[k] = pd.DataFrame(rows).set_index("months ahead")[["n", "rmse", "bias"]]
pd.concat(horizon, axis=1)

In [ ]:
month = holdout["RENT_APPROVAL_DATE"]
by_month = pd.DataFrame({"actual": y_hold, "LightGBM (default)": hold_preds["LightGBM (default)"],
                         "blend (tuned weights)": hold_preds["blend (tuned weights)"]}).groupby(month.to_numpy()).mean()
fig, ax = plt.subplots(figsize=(9, 3.5))
for col, style in [("actual", "-"), ("LightGBM (default)", "--"), ("blend (tuned weights)", "-.")]:
    ax.plot(by_month.index, by_month[col], style, marker="o", ms=3, label=col)
ax.set_ylabel("mean monthly rent (SGD)")
ax.set_title("Holdout months: mean actual vs mean prediction", loc="left")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**D1 findings**

- **Slide 8's horizon problem is unchanged.** Bias is about 0 at 1-3 months ahead and −64 at 13-15, for the
  report's LightGBM and for the blend alike (within 1.4 in every bucket). Tuning and blending tighten the fit
  around the held level; they do not move the level.
- The plot shows it: past 2023-12 the mean prediction stays flat while the actual mean keeps rising slowly.

**Takeaway:** the remaining error is the market level, which no experiment here changes.

### S1. Kaggle dry run (next step 2)

| | |
|---|---|
| **What** | Each E2-tuned booster refit on all of train (2021-01 to 2025-03), its round count scaled to the larger training set as in E2, and blended with E3's weights |
| **Files** | `outputs/submission_09_blend.csv`: the blend, the configuration chosen on the tuning split · `outputs/submission_09_lgbm.csv`: tuned LightGBM alone, best on the holdout (E3) |
| **Format** | Kaggle's: `Id` (0-based row number in the test file) and `Predicted`. `07`'s `outputs/submission.csv` is not overwritten |

Submitting both is a cheap check of whether the leaderboard agrees with the holdout on a 0.5-RMSE difference.

In [ ]:
full_scale = len(train) / len(tune_fit)
test_cols = {}
for name in BOOSTERS:
    params = {**final_params(name, best.loc[name, "rounds"]),
              ROUNDS_KEY[name]: int(round(best.loc[name, "rounds"] * full_scale))}
    test_cols[name] = fit_predict(name, train, test, params)
test_matrix = pd.DataFrame(test_cols)
test_pred = test_matrix.to_numpy() @ W
assert len(test_pred) == len(raw_test) and np.isfinite(test_pred).all()

for path, pred in [(OUT / "submission_09_blend.csv", test_pred), (OUT / "submission_09_lgbm.csv", test_matrix["LightGBM"])]:
    # Kaggle format: Id = 0-based row number in test.csv, Predicted = rent.
    pd.DataFrame({"Id": np.arange(len(pred)), "Predicted": np.asarray(pred)}).to_csv(path, index=False)
    print(f"wrote {path}: {len(pred):,} rows")
print("correlation between the boosters' test predictions:")
display(test_matrix.corr())

test_month = test["RENT_APPROVAL_DATE"].dt.strftime("%Y-%m").to_numpy()
last3 = train[train["MONTH_INDEX"] >= train["MONTH_INDEX"].max() - 2]
print("train, last three months, median rent:", last3.groupby(last3["RENT_APPROVAL_DATE"].dt.strftime("%Y-%m"))[TARGET].median().to_dict())
pd.Series(test_pred, index=test_month).groupby(level=0).median().rename("median prediction").iloc[::3].to_frame()

**S1 notes**

- Both files: 50,000 rows, no missing values, every test town / flat type / flat model seen in train.
- **The test predictions are flat at the end-of-train level:** median 3,130 to 3,165 in every test month, against
  actual medians of 3,100, 3,150 and 3,200 in the last three training months. If rents keep rising through
  2026, expect the same under-prediction as D1 measured (about −60 at 7+ months ahead).
- **Leaderboard:** `submission_09_lgbm.csv` scored 443.25 on the public leaderboard (recorded in `10`). No score
  for `submission_09_blend.csv` is recorded.

### Summary

**Results by experiment** (holdout RMSE, lower is better):

| model | E1: default | E2: tuned |
|---|---|---|
| LightGBM (report: 489.6) | 489.6 | **487.5** |
| XGBoost | 496.7 | 489.6 |
| CatBoost | 498.4 | 490.4 |
| HistGradientBoosting | 490.7 | not tuned |
| Random forest | 503.3 | not tuned |
| Linear regression (report: 652.4) | 652.6 | not tuned |
| E3: blend of the three tuned boosters | | 488.0 |

- **Protocol (next step 1):** fit 2021-01 to 2023-12, score 2024-01 to 2025-03. Tuning, early stopping and blend
  weights use a separate split inside the fit months (fit to 2023-03, score 2023-04 to 2023-12).
- **Other models and tuning (next step 3):** E1, E2, E3 above.
- **Kaggle dry run (next step 2), S1:** `outputs/submission_09_lgbm.csv` (public leaderboard 443.25) and
  `outputs/submission_09_blend.csv`.

**What this says:**

- The report's LightGBM was already close to the best these models can do with these features. Tuning and other
  libraries add up to 2.1 RMSE (E2); blending adds nothing, because the boosters' errors correlate at 0.99 (E3).
- Regularised, shallower trees win in every library, which suits forecasting a market that has moved since the
  fit months.
- The remaining error is the level, not the model (D1): every model here holds the end-of-train level, and
  under-predicts by about 64 at 13-15 months ahead. `07`/`08` (after the report) and `10` test level forecasts;
  that is the direction with room left, not more model tuning.

In [ ]:
print(f"runtime: {(time.time() - T0) / 60:.1f} min")

## Part 2.3. HDB Rentals: block premiums, market level and what else works

Builds on `09`'s tuned LightGBM (holdout 487.5, public leaderboard 443.2) using only the competition and
auxiliary data. **Nine experiments in two phases**, each changing one thing, plus three submission checkpoints.
Two extras that use test-side information are kept in appendices and are not part of the model.

### Protocol (every experiment)

- **`09`'s two splits.** Tuning split: fit 2021-01 to 2023-03, score 2023-04 to 2023-12. Holdout: fit 2021-01 to
  2023-12, score 2024-01 to 2025-03.
- **Choose on the tuning split, report the holdout.** A change is kept only if it lowers the tuning RMSE against
  its reference. Holdout numbers are shown for every row but never pick anything.
- **No leakage.** Premiums, neighbour features and level forecasts are built from each split's fit rows only.
  Leaderboard scores are reported, never used to choose (except Appendix B, which is labelled as such).
- **Each experiment opens with a card** (question, reference, change, variants, what is held fixed, decision
  rule) and closes with its findings and a **Decision** line.

### Experiment log

Δ = RMSE minus the reference's RMSE on the same split; negative is better.

| ID | Experiment | Change | Reference | Tuning Δ | Holdout Δ | Decision |
|---|---|---|---|---|---|---|
| | **Phase 1: features on `09`'s model** | | | | | |
| E0 | Reference | `09`'s tuned LightGBM, unchanged | – | 603.6 | 487.5 | – |
| D1 | Block premium ceiling (diagnostic) | Actual market level × premium at four levels of detail | – | – | 491.6 at best | Expect a few dollars from E1, not tens |
| **E1** | Block + flat type premium | + block, block + flat type premiums and row count; 3 ways to build them | E0 | −3.8 (`oof`) | −4.4 | **Keep `oof`** |
| E2 | Coordinates | + km east/north, optionally rotated copies | E0 | −0.5 | +0.2 | Passes, then dropped in E5 |
| E3 | Neighbouring blocks | + premium of the 10 nearest blocks | E0 | −1.2 | −1.0 | Passes, then dropped in E5 |
| E4 | Recency weighting | Sample weights, half-life 6 to 36 months | E0 | +1.2 to +5.9 | +1.2 to +6.6 | Drop |
| E5 | Combine what passed | E1 + E2 + E3 together | E0 | −3.0 | −4.2 | **E1 alone is final** (beats the combination) |
| S1 | Submission | E5's model refit on all of train | – | – | – | `submission_10_features.csv`: leaderboard **436.8** (from 443.2) |
| | **Phase 2: market level and refinements** | | | | | |
| **E6** | Market level × premium | Time from a forecast market level instead of `MONTH_INDEX` | E5 (599.8 / 483.1) | −5.8 | −1.5 | **Keep `blend` forecast** |
| S2 | Submission | E6 alone, refit on all of train | – | – | – | `submission_10_level.csv`: no score recorded |
| E7 | Premium smoothing | Pull strength of the block premiums | E6 (smoothing 20 / 5) | −0.1 | −0.1 | Keep 5 / 2 (noise-sized) |
| E8 | More premiums | + street, town, flat model or per-sqm premium | E7 | −1.0 | +0.1 | Keep street + flat type |
| E9 | Retune LightGBM | 20 random draws on E6-E8's columns and target | E8 (holdout 481.7) | n/a* | −0.9 | Keep retuned |
| S3 | Final submission | E6 + E7 + E8 + E9, refit on all of train | – | – | – | `submission_10_final.csv`: leaderboard 439.5 (worse than S1) |
| | **Appendices: use test-side information, not valid models** | | | | | |
| A | Look up test rents | data.gov.sg's published records, which contain the test labels | – | – | 188.4 on the check months | `submission_10_external.csv` |
| B | Mix S1 and S3 | Weight computed from their two leaderboard scores | – | – | – | `submission_10_mix.csv`: leaderboard ≈ 434.5 |

\* E9's tuning RMSE is early-stopped on the tuning split itself, so it is not comparable with E8's.

**Contents.** Setup · E0 harness · Phase 1: D1, E1-E5, S1 · Phase 2: E6, S2, E7-E9, S3 · Summary · Appendix A ·
Appendix B

In [ ]:
import json
import time
import urllib.request
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA, EXTERNAL, OUT = ROOT / "data", ROOT / "data" / "external", ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 0
T0 = time.time()

### Setup

Cleaning, `DIST` and the two splits exactly as in `09` (§1 to §3); block coordinates come from the same HDB block
table. Column names are upper case, as in `09`.

- **Holdout:** fit 2021-01 to 2023-12, score 2024-01 to 2025-03.
- **Tuning split:** fit 2021-01 to 2023-03, score 2023-04 to 2023-12.
- **Model:** `09`'s tuned LightGBM, 357 rounds, the same on both splits.
- **Relative rent** = rent ÷ that month's median rent for the flat type. It takes the market level out, so a
  premium of 1.08 means "8% above the typical flat of this type that month". Premiums and neighbour features are
  built only from each split's fit rows; scored rows' rents never enter a feature.

In [ ]:
from scipy import sparse
from sklearn.neighbors import BallTree

CLEANED = ROOT / "cleaned"
TARGET = "MONTHLY_RENT"
KEY = ["TOWN", "BLOCK", "STREET"]


def lower_strip(df):
    out = df.copy()
    text = [c for c in out.columns if pd.api.types.is_string_dtype(out[c])]
    out[text] = out[text].apply(lambda s: s.str.strip().str.lower())
    return out


def clean_rent(df):
    out = lower_strip(df).drop(columns=["FURNISHED", "FEE"])
    out["FLAT_TYPE"] = out["FLAT_TYPE"].str.replace(r"[\s\-]+", "-", regex=True)
    d = pd.to_datetime(out["RENT_APPROVAL_DATE"], format="%Y-%m")
    out["RENT_APPROVAL_DATE"] = d
    out["MONTH_INDEX"] = (d.dt.year - 2021) * 12 + d.dt.month - 1
    out["FLAT_AGE"] = d.dt.year - out["LEASE_COMMENCE_DATE"]  # for the hedonic level in E6
    return out


def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dlat, dlon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dlat / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


tr = clean_rent(pd.read_csv(DATA / "train.csv")).sort_values("MONTH_INDEX", kind="stable").reset_index(drop=True)
te = clean_rent(pd.read_csv(DATA / "test.csv"))

hdb = pd.read_csv(CLEANED / "sg_hdb_block_cleaned.csv", dtype={"POSTAL_CODE": str})
mrt = pd.read_csv(CLEANED / "sg_mrt_stations_cleaned.csv")
schools = pd.read_csv(CLEANED / "sg_schools_cleaned.csv")
malls = lower_strip(pd.read_csv(DATA / "auxiliary" / "sg-shopping-malls.csv"))
with open(DATA / "auxiliary" / "HawkerCentresGEOJSON.geojson") as f:
    geo = json.load(f)["features"]
hawker = lower_strip(pd.DataFrame({
    "NAME": [g["properties"]["NAME"] for g in geo],
    "STATUS": [g["properties"]["STATUS"] for g in geo],
    "LONGITUDE": [g["geometry"]["coordinates"][0] for g in geo],
    "LATITUDE": [g["geometry"]["coordinates"][1] for g in geo],
}))
mrt_open = mrt[mrt["STATUS"].eq("open")].drop_duplicates("NAME")
POI_GROUPS = {
    "primary": schools[schools["MAINLEVEL_CODE"].str.contains("primary", na=False)],
    "secondary": schools[schools["MAINLEVEL_CODE"].str.contains("secondary", na=False)],
    "mrt": mrt_open,
    "mrt_interchange": mrt_open[mrt_open["IS_INTERCHANGE"]],
    "malls": malls,
    "hawker": hawker[hawker["STATUS"].ne("under construction")],
}
lat, lon = hdb["LATITUDE"].to_numpy(), hdb["LONGITUDE"].to_numpy()
for prefix, poi in POI_GROUPS.items():
    d = haversine_km(lat[:, None], lon[:, None], poi["LATITUDE"].to_numpy()[None, :], poi["LONGITUDE"].to_numpy()[None, :])
    hdb[f"{prefix}_nearest_km"] = d.min(axis=1)
    for label, r in {"1km": 1.0, "2km": 2.0}.items():
        hdb[f"{prefix}_within_{label}"] = (d <= r).sum(axis=1)
cbd = mrt.loc[mrt["NAME"].eq("raffles place mrt station"), ["LATITUDE", "LONGITUDE"]].iloc[0]
hdb["dist_to_cbd_km"] = haversine_km(lat, lon, cbd["LATITUDE"], cbd["LONGITUDE"])

DIST = [
    "MAX_FLOOR", "primary_within_2km", "mrt_within_2km",
    "primary_nearest_km", "mrt_nearest_km", "secondary_within_1km",
    "mrt_interchange_within_1km", "mrt_interchange_nearest_km",
    "malls_within_1km", "malls_nearest_km", "hawker_within_1km", "hawker_nearest_km", "dist_to_cbd_km",
]
n_tr, n_te = len(tr), len(te)
tr = tr.merge(hdb[KEY + DIST + ["LATITUDE", "LONGITUDE"]], on=KEY, how="left", validate="m:1")
te = te.merge(hdb[KEY + DIST + ["LATITUDE", "LONGITUDE"]], on=KEY, how="left", validate="m:1")
assert len(tr) == n_tr and len(te) == n_te and tr["LATITUDE"].notna().all() and te["LATITUDE"].notna().all()

In [ ]:
# One id per block (town + block + street) and per block + flat type, shared by train and test.
BLOCKS = pd.concat([tr, te])[KEY + ["LATITUDE", "LONGITUDE"]].drop_duplicates(KEY).reset_index(drop=True)
BLOCKS["BID"] = np.arange(len(BLOCKS))
FT_LEVELS = sorted(tr["FLAT_TYPE"].unique())
NB, NFT = len(BLOCKS), len(FT_LEVELS)


def add_ids(df):
    out = df.merge(BLOCKS[KEY + ["BID"]], on=KEY, how="left", validate="m:1")
    out["FT"] = pd.Categorical(out["FLAT_TYPE"], categories=FT_LEVELS).codes
    out["G"] = out["BID"] * NFT + out["FT"]
    return out


tr, te = add_ids(tr), add_ids(te)
assert (te["FT"] >= 0).all()
tr["REL"] = tr[TARGET] / tr.groupby(["MONTH_INDEX", "FLAT_TYPE"])[TARGET].transform("median")
# The same per square metre, for E8's per-sqm premium.
rent_sqm = tr[TARGET] / tr["FLOOR_AREA_SQM"]
tr["REL_SQM"] = rent_sqm / rent_sqm.groupby([tr["MONTH_INDEX"], tr["FLAT_TYPE"]]).transform("median")

BASE_NUM = ["FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE", "MONTH_INDEX"] + DIST
CAT = ["TOWN", "FLAT_TYPE", "FLAT_MODEL"]
CATEGORIES = {c: pd.CategoricalDtype(sorted(tr[c].unique())) for c in CAT}
LGBM_TUNED = {"num_leaves": 63, "min_child_samples": 100, "subsample": 0.85, "subsample_freq": 1,
              "colsample_bytree": 0.5, "reg_lambda": 20.0, "cat_smooth": 30.0, "n_estimators": 357,
              "learning_rate": 0.05}


def matrix(df, extra=(), time=True):
    num = BASE_NUM if time else [c for c in BASE_NUM if c != "MONTH_INDEX"]
    X = df[num + list(extra) + CAT].copy()
    for c in CAT:
        X[c] = X[c].astype(CATEGORIES[c])
    return X


def fit_lgbm(fit_df, extra=(), weight=None, **overrides):
    est = lgb.LGBMRegressor(**{**LGBM_TUNED, **overrides}, verbose=-1, n_jobs=-1, random_state=SEED)
    return est.fit(matrix(fit_df, extra), fit_df[TARGET].to_numpy(), sample_weight=weight)


def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(p, float)) ** 2)))


SPLITS = {"tuning": ("2023-04", "2024-01"), "holdout": ("2024-01", "2099-01")}
FRAMES = {}
for s, (start, end) in SPLITS.items():
    d = tr["RENT_APPROVAL_DATE"]
    FRAMES[s] = [tr[d < start].reset_index(drop=True), tr[(d >= start) & (d < end)].reset_index(drop=True)]
    print(f"{s:<8} fit {len(FRAMES[s][0]):>7,} rows, score {len(FRAMES[s][1]):>6,} rows")
print(f"blocks: {NB:,}  block + flat type pairs seen in train: {tr['G'].nunique():,}")

### E0. Experiment harness and reference

`run(name, extra, half_life)` fits the model on each split's fit rows with the base columns plus `extra` (and
optional recency weights) and records tuning and holdout RMSE and bias in `RESULTS`. `results_table` shows rows
against **E0**, `09`'s tuned LightGBM with no extra columns. Phase 1 (D1 to E5) uses this harness.

In [ ]:
RESULTS = {}


def run(name, extra=(), half_life=None):
    row = {}
    for s, (fit_df, val_df) in FRAMES.items():
        m = fit_df["MONTH_INDEX"].to_numpy()
        w = None if half_life is None else 0.5 ** ((m.max() - m) / half_life)
        p = fit_lgbm(fit_df, extra, w).predict(matrix(val_df, extra))
        y = val_df[TARGET].to_numpy()
        row[f"{s} RMSE"], row[f"{s} bias"] = rmse(y, p), float(np.mean(p - y))
    RESULTS[name] = {**row, "extra": list(extra), "half_life": half_life}
    return row


def results_table(names):
    t = pd.DataFrame({n: RESULTS[n] for n in names}).T[["tuning RMSE", "holdout RMSE", "tuning bias", "holdout bias"]]
    t = t.astype(float)
    for s in ["tuning", "holdout"]:
        t[f"{s} vs base"] = t[f"{s} RMSE"] - RESULTS["base (09 tuned LightGBM)"][f"{s} RMSE"]
    return t[["tuning RMSE", "tuning vs base", "holdout RMSE", "holdout vs base", "tuning bias", "holdout bias"]]


run("base (09 tuned LightGBM)")
results_table(["base (09 tuned LightGBM)"])

## Phase 1: features on top of `09`'s model

Reference for every experiment in this phase: **E0** (tuning 603.6, holdout 487.5).

### D1. How much can a block's own past rents tell? (diagnostic)

| | |
|---|---|
| **Question** | Before building any feature: if the market level were known exactly, how good is a premium at each level of detail? |
| **Method** | Predict each holdout row as (actual median rent for its month and flat type) × (premium from the fit months) |
| **Varied** | Premium detail: none · town + flat type · block · block + flat type |
| **Caveat** | Uses the holdout's actual level, so these are not achievable scores. They isolate the premium part and show which level of detail carries information |
| **Used for** | Setting expectations for E1 |

Premiums are smoothed averages of relative rent: a group with few rows is pulled toward the level above it
(block + flat type toward block with weight 5 rows, block toward 1 with weight 20, town + flat type toward 1 with
weight 20).

In [ ]:
M_BLOCK, M_FT = 20, 5


def encode(src, dst, w, m=(M_BLOCK, M_FT)):
    # Smoothed block and block + flat type premiums from src rows (weights w), looked up for dst rows.
    m_block, m_ft = m
    bid, g, rel = src["BID"].to_numpy(), src["G"].to_numpy(), src["REL"].to_numpy()
    sb, cb = np.bincount(bid, w * rel, NB), np.bincount(bid, w, NB)
    sg, cg = np.bincount(g, w * rel, NB * NFT), np.bincount(g, w, NB * NFT)
    n = np.bincount(g, minlength=NB * NFT)
    db, dg = dst["BID"].to_numpy(), dst["G"].to_numpy()
    pb = (m_block + sb[db]) / (m_block + cb[db])
    pg = (m_ft * pb + sg[dg]) / (m_ft + cg[dg])
    return np.column_stack([pb, pg, n[dg]])


fit_h, hold_h = FRAMES["holdout"]
y_h = hold_h[TARGET].to_numpy()
oracle_level = hold_h.groupby(["MONTH_INDEX", "FLAT_TYPE"])[TARGET].transform("median").to_numpy()
town_ft = fit_h.groupby(["TOWN", "FLAT_TYPE"])["REL"].agg(["sum", "size"])
town_ft_prem = ((M_BLOCK + town_ft["sum"]) / (M_BLOCK + town_ft["size"])).rename("p")
p_town = hold_h.join(town_ft_prem, on=["TOWN", "FLAT_TYPE"])["p"].fillna(1).to_numpy()
p_block, p_block_ft, n_seen = encode(fit_h, hold_h, np.ones(len(fit_h))).T
seen = n_seen > 0

check_tbl = pd.DataFrame({
    "flat type only (premium = 1)": [rmse(y_h, oracle_level)],
    "town + flat type": [rmse(y_h, oracle_level * p_town)],
    "block": [rmse(y_h, oracle_level * p_block)],
    "block + flat type": [rmse(y_h, oracle_level * p_block_ft)],
}, index=["holdout RMSE with the actual level"]).T
check_tbl["block + flat type seen in fit (" + f"{seen.mean():.0%}" + " of rows)"] = [
    rmse(y_h[seen], (oracle_level * p)[seen]) for p in [np.ones(len(y_h)), p_town, p_block, p_block_ft]]
display(check_tbl)
print("for reference, 09's tuned LightGBM on the holdout: 487.5 (it has to forecast the level too)")

**D1 findings**

- **Even with the actual market level, the block + flat type premium only reaches 491.6**, about where `09`'s model already is (487.5) while also having to forecast the level. Detail helps in steps (flat type only 567.7, town + flat type 516.0, block 511.0, block + flat type 491.6), but the last step is worth about 25, and the model's town, flat model, floor area and `DIST` columns already cover much of it.
- **99% of holdout rows have their block + flat type in the fit months**, so this is not a coverage problem: the same block's flats of one type just rent at quite different prices from one year to the next.

**Takeaway:** block history is not worth 100+ RMSE. Expect a gain of a few dollars from E1, not tens.

### E1. Block + flat type premium

| | |
|---|---|
| **Question** | Does a block's rent history, relative to the market, add information the model lacks? |
| **Reference** | E0 |
| **Change** | + 3 columns: block premium, block + flat type premium, and how many fit rows the block + flat type had (so the tree can tell a premium from 30 rentals from one built on 1) |
| **Variants** | How each fit row gets its own premium: `oof` · `past` · `recent` (below) |
| **Held fixed** | Model settings (357 rounds), base columns, smoothing 20 / 5, splits. Scored and test rows always use all fit months |
| **Decision rule** | Keep the variant with the lowest tuning RMSE if it beats E0 |

The variants differ only in what the **fit rows** see:

- **`oof` (out-of-fold):** each fit row's premium comes from the other 4/5 of the fit rows, over all months (as
  `08` did for blocks).
- **`past` (past only):** each fit row's premium comes only from earlier months. This is how a test row sees the
  data, so the tree learns how much to trust a premium as it would be at prediction time.
- **`recent` (past only, recency-weighted):** as `past`, with each earlier rental's weight halving every 12 months,
  so the premium follows a block that is drifting.

In [ ]:
PREMIUM_VARIANTS = {"oof": ("oof", None), "past": ("past", None), "recent": ("past", 12)}


def premiums(fit_df, apply_dfs, mode, half_life=None, m=(M_BLOCK, M_FT)):
    month = fit_df["MONTH_INDEX"].to_numpy()
    weights = lambda m, ref: np.ones(len(m)) if half_life is None else 0.5 ** ((ref - m) / half_life)
    T = month.max()
    applied = [encode(fit_df, a, weights(month, T), m) for a in apply_dfs]
    own = np.empty((len(fit_df), 3))
    if mode == "oof":
        for f_idx, p_idx in KFold(5, shuffle=True, random_state=SEED).split(fit_df):
            src = fit_df.iloc[f_idx]
            own[p_idx] = encode(src, fit_df.iloc[p_idx], weights(month[f_idx], T), m)
    else:
        for t in np.unique(month):
            before, now = month < t, month == t
            own[now] = encode(fit_df[before], fit_df[now], weights(month[before], t - 1), m)
    return own, applied


def premium_cols(tag):
    return [f"BLOCK_PREM_{tag}", f"BLOCK_FT_PREM_{tag}", f"BLOCK_FT_N_{tag}"]


for s, (fit_df, val_df) in FRAMES.items():
    for tag, (mode, hl) in PREMIUM_VARIANTS.items():
        own, (applied,) = premiums(fit_df, [val_df], mode, hl)
        fit_df[premium_cols(tag)], val_df[premium_cols(tag)] = own, applied

for tag in PREMIUM_VARIANTS:
    run(f"+ block premiums, {tag}", premium_cols(tag))
results_table(["base (09 tuned LightGBM)"] + [f"+ block premiums, {t}" for t in PREMIUM_VARIANTS])

**E1 findings**

- **The out-of-fold premiums help a little on both splits:** tuning 603.6 → 599.8 (−3.8), holdout 487.5 → 483.1 (−4.4). It is the largest gain any single change has made since the report's `DIST` set.
- **Past only is close behind** (−2.5 / −4.2). **Recency weighting the premium is worse than either** (−1.7 / −2.1): with few rentals per block + flat type, down-weighting older ones mostly adds noise.
- Bias barely moves (−94 vs −89 on tuning, −37 vs −35 on holdout): premiums sharpen who is above or below the market, not where the market goes.

**Decision: keep `oof`.** It goes to E5.

### E2. Coordinates

| | |
|---|---|
| **Question** | Does raw position help beyond `TOWN` and `DIST`, including along diagonal boundaries? |
| **Reference** | E0 |
| **Change** | + `X_KM`, `Y_KM`: each block's position in km east and north of the island's centre |
| **Variants** | plain (2 columns) · rotated (+ copies rotated by 15°, 30°, 45°, 60° and 75°: 7 columns) |
| **Held fixed** | As E1 |
| **Decision rule** | Keep the variant with the lowest tuning RMSE if it beats E0 |

Why rotated copies: trees split one column at a time, so a diagonal boundary (a river, an expressway) needs many
splits; the rotated columns give the tree those diagonals directly.

In [ ]:
LAT0, LON0 = 1.35, 103.82
BLOCKS["X_KM"] = (BLOCKS["LONGITUDE"] - LON0) * 111.32 * np.cos(np.radians(LAT0))
BLOCKS["Y_KM"] = (BLOCKS["LATITUDE"] - LAT0) * 110.57
ANGLES = [15, 30, 45, 60, 75]
for a in ANGLES:
    r = np.radians(a)
    BLOCKS[f"ROT_{a}"] = BLOCKS["X_KM"] * np.cos(r) + BLOCKS["Y_KM"] * np.sin(r)
COORDS = ["X_KM", "Y_KM"]
ROTATED = COORDS + [f"ROT_{a}" for a in ANGLES]

for fit_df, val_df in FRAMES.values():
    for df in (fit_df, val_df):
        df[ROTATED] = BLOCKS.loc[df["BID"], ROTATED].to_numpy()

run("+ coordinates", COORDS)
run("+ coordinates, rotated", ROTATED)
results_table(["base (09 tuned LightGBM)", "+ coordinates", "+ coordinates, rotated"])

**E2 findings**

- **Coordinates do nothing:** −0.5 on tuning, +0.2 on holdout; rotated copies are no better (−0.2 / +0.3).
  `TOWN` and the 13 `DIST` columns already pin down location.

**Decision:** plain coordinates pass the rule (by 0.5 on tuning), so they go to E5; rotated copies do not.

### E3. Neighbouring blocks

| | |
|---|---|
| **Question** | Do nearby blocks' rents say something about a block's own? (The cheap version of a graph model: "blocks near each other rent alike") |
| **Reference** | E0 |
| **Change** | + 2 columns: the pooled premium of the 10 nearest other blocks for the row's flat type, and over all flat types, smoothed as in E1 |
| **Variants** | neighbours alone · neighbours + E2's rotated coordinates |
| **Held fixed** | As E1. Built from fit rows only, and a row's own block is never in its neighbour set, so its own rent never enters |
| **Decision rule** | Keep if it beats E0 on the tuning split |

In [ ]:
K_NEIGHBOURS = 10
pos = np.radians(BLOCKS[["LATITUDE", "LONGITUDE"]].to_numpy())
dist, idx = BallTree(pos, metric="haversine").query(pos, k=K_NEIGHBOURS + 1)
src_rows = np.repeat(np.arange(NB), K_NEIGHBOURS + 1)
keep = idx.ravel() != src_rows
ADJ = sparse.csr_matrix((np.ones(keep.sum()), (src_rows[keep], idx.ravel()[keep])), shape=(NB, NB))
print(f"median distance to the 10th nearest block: {np.median(dist[:, -1]) * 6371:.2f} km")
NEIGHBOUR = ["NBR_FT_PREM", "NBR_PREM"]


def neighbour_premiums(fit_df, dfs):
    g, rel = fit_df["G"].to_numpy(), fit_df["REL"].to_numpy()
    sg = np.bincount(g, rel, NB * NFT).reshape(NB, NFT)
    cg = np.bincount(g, minlength=NB * NFT).reshape(NB, NFT).astype(float)
    ns, nc = ADJ @ sg, ADJ @ cg
    p_all = (M_BLOCK + ns.sum(1)) / (M_BLOCK + nc.sum(1))
    p_ft = (M_FT * p_all[:, None] + ns) / (M_FT + nc)
    return [np.column_stack([p_ft[d["BID"].to_numpy(), d["FT"].to_numpy()], p_all[d["BID"].to_numpy()]]) for d in dfs]


for fit_df, val_df in FRAMES.values():
    fit_df[NEIGHBOUR], val_df[NEIGHBOUR] = neighbour_premiums(fit_df, [fit_df, val_df])

run("+ neighbours", NEIGHBOUR)
run("+ coordinates, rotated + neighbours", ROTATED + NEIGHBOUR)
results_table(["base (09 tuned LightGBM)", "+ neighbours", "+ coordinates, rotated + neighbours"])

**E3 findings**

- **Neighbours help slightly and consistently:** −1.2 tuning, −1.0 holdout. The 10 nearest blocks are within about 170 m, so this is a very local premium. Adding rotated coordinates on top does not help on the tuning split (−0.9).
- A graph network would learn the same "nearby blocks rent alike" signal; since the cheap version is worth about 1, a graph network is not worth building for this data.

**Decision:** neighbours alone pass and go to E5.

### E4. Weighting recent months

| | |
|---|---|
| **Question** | Does weighting recent months more help the model follow the market? |
| **Reference** | E0 |
| **Change** | Each training row is weighted 0.5^(months before the last fit month ÷ half-life). A half-life of 12 means a row from a year before the cutoff counts half as much as one from the last month |
| **Variants** | half-life 6, 12, 24, 36 months |
| **Held fixed** | Base columns (no premiums), settings, splits |
| **Decision rule** | Keep the best half-life if it beats E0 on the tuning split |

In [ ]:
HALF_LIVES = [6, 12, 24, 36]
for hl in HALF_LIVES:
    run(f"weighted, half-life {hl}", half_life=hl)
results_table(["base (09 tuned LightGBM)"] + [f"weighted, half-life {hl}" for hl in HALF_LIVES])

**E4 findings**

- **Weighting recent months hurts at every half-life:** +5.9 / +6.6 at 6 months, still +1.2 / +1.2 at 24 to 36 months. Older rows still teach the tree how flats differ from each other, and dropping their weight loses that more than it helps follow the recent market.
- The bias does not improve either (−90 on tuning, −32 to −34 on holdout), so weighting does not fix the under-prediction far ahead.

**Decision: drop.** Nothing goes to E5.

### E5. Everything that passed, together

| | |
|---|---|
| **Question** | Do the changes that passed on their own add up? |
| **Reference** | E0, and each part alone |
| **Change** | From each family (E1 premium variant, E2 coordinates, E3 neighbours, E4 weighting), the variant with the lowest tuning RMSE, if it beats E0 there |
| **Decision rule** | The final Phase 1 model is whichever of the combination and its parts has the lowest tuning RMSE (combining can do worse than its best part) |

The second cell breaks the holdout down by months ahead for E0 and the final model.

In [ ]:
FAMILIES = {
    "premium": [f"+ block premiums, {t}" for t in PREMIUM_VARIANTS],
    "coordinates": ["+ coordinates", "+ coordinates, rotated"],
    "neighbours": ["+ neighbours"],
    "weighting": [f"weighted, half-life {hl}" for hl in HALF_LIVES],
}
base_tune = RESULTS["base (09 tuned LightGBM)"]["tuning RMSE"]
chosen = {}
for fam, names in FAMILIES.items():
    best_name = min(names, key=lambda n: RESULTS[n]["tuning RMSE"])
    if RESULTS[best_name]["tuning RMSE"] < base_tune:
        chosen[fam] = best_name
print("kept:", chosen)

combo_extra = [c for n in chosen.values() for c in RESULTS[n]["extra"]]
combo_hl = RESULTS[chosen["weighting"]]["half_life"] if "weighting" in chosen else None
run("combined", combo_extra, combo_hl)
# Combining can do worse than its best part, so the final model is whichever is best on the tuning split.
final = min(["combined"] + list(chosen.values()), key=lambda n: RESULTS[n]["tuning RMSE"])
final_extra, final_hl = RESULTS[final]["extra"], RESULTS[final]["half_life"]
print("final model:", final)
results_table(["base (09 tuned LightGBM)"] + list(chosen.values()) + ["combined"])

In [ ]:
# Error by months ahead on the holdout, base vs the final model.
fit_df, val_df = FRAMES["holdout"]
m = fit_df["MONTH_INDEX"].to_numpy()
w = None if final_hl is None else 0.5 ** ((m.max() - m) / final_hl)
p_base = fit_lgbm(fit_df).predict(matrix(val_df))
p_combo = fit_lgbm(fit_df, final_extra, w).predict(matrix(val_df, final_extra))
ahead = val_df["MONTH_INDEX"] - m.max()
bucket = pd.cut(ahead, [0, 3, 6, 9, 12, 15], labels=["1-3", "4-6", "7-9", "10-12", "13-15"])
y = val_df[TARGET].to_numpy()
pd.DataFrame({
    "base RMSE": pd.Series((p_base - y) ** 2).groupby(bucket, observed=True).mean() ** 0.5,
    "final RMSE": pd.Series((p_combo - y) ** 2).groupby(bucket, observed=True).mean() ** 0.5,
    "base bias": pd.Series(p_base - y).groupby(bucket, observed=True).mean(),
    "final bias": pd.Series(p_combo - y).groupby(bucket, observed=True).mean(),
}).rename_axis("months ahead")

**E5 findings**

- **Adding coordinates and neighbours on top of the premiums makes it worse on the tuning split** (600.6 vs 599.8).
- **The final model is better than E0 in every months-ahead bucket** (2 to 5 RMSE each), and the far-ahead under-prediction (−63 to −65 at 10 to 15 months) is untouched: that is the level problem from `07`/`08`, which no feature here addresses.

**Decision: the Phase 1 model is E0 + E1's `oof` block premiums.** It is the reference for Phase 2.

#### S1. Submission checkpoint: E5's model

E5's model refit on all of train, with rounds scaled to the larger training set as `09` did
(357 × all train ÷ holdout fit). Train rows get the chosen premium variant; test rows use all of train. E0 is
refit alongside as a check that it reproduces `submission_09_lgbm.csv`.

In [ ]:
# Refit on all of train. Premiums for train rows use the chosen variant; test rows use all of train.
full_tr, full_te = tr.copy(), te.copy()
for df in (full_tr, full_te):
    df[ROTATED] = BLOCKS.loc[df["BID"], ROTATED].to_numpy()
full_tr[NEIGHBOUR], full_te[NEIGHBOUR] = neighbour_premiums(full_tr, [full_tr, full_te])
for tag, (mode, hl) in PREMIUM_VARIANTS.items():
    own, (applied,) = premiums(full_tr, [full_te], mode, hl)
    full_tr[premium_cols(tag)], full_te[premium_cols(tag)] = own, applied

rounds = int(round(LGBM_TUNED["n_estimators"] * len(full_tr) / len(FRAMES["holdout"][0])))
m = full_tr["MONTH_INDEX"].to_numpy()
w = None if final_hl is None else 0.5 ** ((m.max() - m) / final_hl)
test_base = fit_lgbm(full_tr, n_estimators=rounds).predict(matrix(full_te))
test_combo = fit_lgbm(full_tr, final_extra, w, n_estimators=rounds).predict(matrix(full_te, final_extra))

prev = pd.read_csv(OUT / "submission_09_lgbm.csv")["Predicted"].to_numpy()
print(f"rounds on all of train: {rounds}")
print(f"base refit vs submission_09_lgbm.csv (should be ~0, same model): RMSE {rmse(prev, test_base):.2f}")
print(f"final vs base on test: RMSE between them {rmse(test_base, test_combo):.1f}, "
      f"mean shift {np.mean(test_combo - test_base):+.1f}")
assert len(test_combo) == len(te) and np.isfinite(test_combo).all()
SUB_FEATURES = OUT / "submission_10_features.csv"
pd.DataFrame({"Id": np.arange(len(test_combo)), "Predicted": test_combo}).to_csv(SUB_FEATURES, index=False)
print(f"wrote {SUB_FEATURES}")

**S1 notes**

- **File:** `outputs/submission_10_features.csv`, **public leaderboard 436.8** (from 443.2).
- The E0 refit reproduces `submission_09_lgbm.csv` to within 0.9 RMSE, and the new predictions move by 73 RMSE from it with almost no mean shift (+2.3). Locally that is worth about −4.

## Phase 2: market level and refinements

Reference for E6: **E5's model** (E0 + `oof` block premiums; tuning 599.8, holdout 483.1). From E7 on, each
experiment's reference is the previous experiment's chosen setup.

### E6. Market level × premium

| | |
|---|---|
| **Question** | Does forecasting the market level explicitly beat letting the tree hold the last level through `MONTH_INDEX`? |
| **Reference** | E5's model (time from `MONTH_INDEX`) |
| **Change** | rent ≈ market level(month) × premium(flat, location). LightGBM learns the premium; `MONTH_INDEX` is dropped |
| **Variants** | How the level is carried past the last fit month: `hold` · `damped` · `blend` |
| **Prior from `08`** | Only `blend` is eligible: at nine cutoffs `damped` blew up once (over-predicting by 533 right after the 2022 climb ended) and `blend` was best on both the mean and the worst cutoff. `hold` and `damped` are shown for information |
| **Held fixed** | E5's columns (base + `oof` block premiums), settings (357 rounds), splits |
| **Decision rule** | Keep `blend` if it beats `MONTH_INDEX` on the tuning split |

Why: a tree can't extrapolate `MONTH_INDEX`, so past the last training month it holds the last level, and on
the holdout it under-predicts by about 60 at 10 to 15 months ahead (E5). `07` §6 and `08` §4 built the
alternative; it is repeated here with E1's block premiums added.

- **Level:** month effects from an ordinary least squares fit of log rent on month dummies plus town, flat type,
  flat model, floor area and flat age, refit on each split's fit months only (`08` §4).
- **Premium:** LightGBM learns rent ÷ level, with `MONTH_INDEX` dropped. Rows are weighted by level², so the
  squared error it minimises is the squared error in dollars.
- **Forecast past the last fit month:** `hold` repeats the last level, `damped` carries a damped trend (Holt's
  method), `blend` averages the two in log space.

The first cell defines the level model and the Phase 2 harness (`fit_arm`, `run2`, `table2`, results in `R2`),
used by E6 to E9. The second runs E6; the third breaks the holdout down by months ahead.

In [ ]:
import warnings

from statsmodels.tsa.holtwinters import ExponentialSmoothing

HEDONIC_CAT, HEDONIC_NUM = ["TOWN", "FLAT_TYPE", "FLAT_MODEL"], ["FLOOR_AREA_SQM", "FLAT_AGE"]


def hedonic_level(df):
    # Log market level per MONTH_INDEX, relative to the first month (08 §4).
    months = df["MONTH_INDEX"]
    X = pd.concat([
        pd.get_dummies(months, prefix="m", drop_first=True, dtype=float),
        pd.get_dummies(df[HEDONIC_CAT], drop_first=True, dtype=float),
        df[HEDONIC_NUM].astype(float),
    ], axis=1)
    X.insert(0, "const", 1.0)
    coef, *_ = np.linalg.lstsq(X.to_numpy(), np.log(df[TARGET].to_numpy()), rcond=None)
    coef = pd.Series(coef, index=X.columns)
    ordered = np.sort(months.unique())
    return pd.Series([0.0] + [coef[f"m_{m}"] for m in ordered[1:]], index=ordered)


def forecast_level(level, months, method):
    ahead = np.sort(np.unique(months[months > level.index.max()]))
    hold = np.full(len(ahead), level.iloc[-1])
    if method == "hold":
        future = hold
    else:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            fit = ExponentialSmoothing(level.to_numpy(), trend="add", damped_trend=True,
                                       initialization_method="estimated").fit()
        damped = fit.forecast(len(ahead))
        future = damped if method == "damped" else (hold + damped) / 2
    return pd.concat([level, pd.Series(future, index=ahead)])


def level_scales(fit_df, apply_dfs, method):
    months = np.concatenate([a["MONTH_INDEX"].to_numpy() for a in apply_dfs])
    level = forecast_level(hedonic_level(fit_df), months, method)
    to_scale = lambda df: np.exp(level.reindex(df["MONTH_INDEX"]).to_numpy())
    return to_scale(fit_df), [to_scale(a) for a in apply_dfs]


def fit_arm(fit_df, apply_dfs, extra, arm, params=None, early_stop=False):
    # Fit LightGBM on rent with MONTH_INDEX ("MONTH_INDEX") or on rent / level ("level-<method>"), and predict
    # rent for each frame in apply_dfs. With early_stop, apply_dfs[0] is the early-stopping set and the
    # number of rounds used is returned too.
    params = {**LGBM_TUNED, **(params or {})}
    time_col = arm == "MONTH_INDEX"
    if time_col:
        s_fit, s_apply = np.ones(len(fit_df)), [np.ones(len(a)) for a in apply_dfs]
    else:
        s_fit, s_apply = level_scales(fit_df, apply_dfs, arm.split("-", 1)[1])
    est = lgb.LGBMRegressor(**params, verbose=-1, n_jobs=-1, random_state=SEED)
    kw = {}
    if early_stop:
        e = apply_dfs[0]
        kw = {"eval_X": matrix(e, extra, time_col), "eval_y": e[TARGET].to_numpy() / s_apply[0],
              "eval_sample_weight": [s_apply[0] ** 2], "callbacks": [lgb.early_stopping(100, verbose=False)]}
    est.fit(matrix(fit_df, extra, time_col), fit_df[TARGET].to_numpy() / s_fit, sample_weight=s_fit ** 2, **kw)
    preds = [est.predict(matrix(a, extra, time_col)) * s for a, s in zip(apply_dfs, s_apply)]
    return (preds, est.best_iteration_) if early_stop else preds


R2 = {}


def run2(name, extra, arm, params=None):
    row = {}
    for s, (fit_df, val_df) in FRAMES.items():
        (p,) = fit_arm(fit_df, [val_df], extra, arm, params)
        y = val_df[TARGET].to_numpy()
        row[f"{s} RMSE"], row[f"{s} bias"] = rmse(y, p), float(np.mean(p - y))
    R2[name] = {**row, "extra": list(extra), "arm": arm}
    return row


def table2(names, ref):
    t = pd.DataFrame({n: R2[n] for n in names}).T[["tuning RMSE", "holdout RMSE", "tuning bias", "holdout bias"]].astype(float)
    for s in ["tuning", "holdout"]:
        t[f"{s} vs ref"] = t[f"{s} RMSE"] - R2[ref][f"{s} RMSE"]
    return t[["tuning RMSE", "tuning vs ref", "holdout RMSE", "holdout vs ref", "tuning bias", "holdout bias"]]

In [ ]:
ARMS = ["MONTH_INDEX", "level-hold", "level-damped", "level-blend"]
for arm in ARMS:
    run2(f"E5 final, {arm}", final_extra, arm)
REF = "E5 final, MONTH_INDEX"
ARM = "level-blend" if R2["E5 final, level-blend"]["tuning RMSE"] < R2[REF]["tuning RMSE"] else "MONTH_INDEX"
print(f"setup carried forward: {ARM}  (reference row = E5's model, S1 leaderboard 436.8)")
table2([f"E5 final, {a}" for a in ARMS], REF)

In [ ]:
# Error by months ahead on the holdout, MONTH_INDEX vs blend.
fit_df, val_df = FRAMES["holdout"]
(p_mi,) = fit_arm(fit_df, [val_df], final_extra, "MONTH_INDEX")
(p_bl,) = fit_arm(fit_df, [val_df], final_extra, "level-blend")
y = val_df[TARGET].to_numpy()
bucket = pd.cut(val_df["MONTH_INDEX"] - fit_df["MONTH_INDEX"].max(), [0, 3, 6, 9, 12, 15],
                labels=["1-3", "4-6", "7-9", "10-12", "13-15"])
grp = lambda v: pd.Series(v).groupby(bucket, observed=True)
pd.DataFrame({"MONTH_INDEX RMSE": grp((p_mi - y) ** 2).mean() ** 0.5, "blend RMSE": grp((p_bl - y) ** 2).mean() ** 0.5,
              "MONTH_INDEX bias": grp(p_mi - y).mean(), "blend bias": grp(p_bl - y).mean()}).rename_axis("months ahead")

**E6 findings**

- **The blend beats `MONTH_INDEX` on both splits:** tuning 599.8 → 594.0 (−5.8), holdout 483.1 → 481.7 (−1.5).
- **It fixes the far-ahead under-prediction:** bias at 10 to 15 months goes from −63/−65 to −13/−9, and RMSE there drops 4. The price is a small over-prediction in the first 6 months (+23/+24 bias vs about 0), which is why the overall holdout gain is only 1.5.
- **Damped alone looks best on the tuning split (−8.6)** but over-predicts on the holdout (+26 bias); `08` showed it can over-predict by 500+ right after a climb ends. Hold changes little. The blend sits between them on both splits, as in `08`.

**Decision: keep the `blend` level.** It is the setup for E7 to E9.

#### S2. Submission checkpoint: E6 alone

E6's setup on E5's columns, refit on all of train, so the leaderboard can tell the level change apart from E7 to E9.

In [ ]:
# S2: E6 alone, so the leaderboard can tell the level change apart from E7 to E9.
full_tr, full_te = tr.copy(), te.copy()
for tag, (mode, hl) in PREMIUM_VARIANTS.items():
    own, (applied,) = premiums(full_tr, [full_te], mode, hl)
    full_tr[premium_cols(tag)], full_te[premium_cols(tag)] = own, applied
ROUNDS_ALL = int(round(LGBM_TUNED["n_estimators"] * len(full_tr) / len(FRAMES["holdout"][0])))
(test_level,) = fit_arm(full_tr, [full_te], final_extra, ARM, {"n_estimators": ROUNDS_ALL})
SUB_LEVEL = OUT / "submission_10_level.csv"
pd.DataFrame({"Id": np.arange(len(test_level)), "Predicted": test_level}).to_csv(SUB_LEVEL, index=False)
print(f"wrote {SUB_LEVEL} ({ARM}); vs submission_10_features.csv: RMSE between them "
      f"{rmse(test_level, test_combo):.1f}, mean shift {np.mean(test_level - test_combo):+.1f}")
blend_level = forecast_level(hedonic_level(full_tr), full_te["MONTH_INDEX"].to_numpy(), "blend")
print(f"blend level, 2025-03 to 2026-07: {np.exp(blend_level.iloc[-1] - blend_level.loc[50]) - 1:+.1%}")

**S2 notes**

- **File:** `outputs/submission_10_level.csv`. No leaderboard score is recorded for it.
- **On test, the blend raises predictions by about 90 on average** over S1 (its level rises 2.1% from 2025-03 to 2026-07, and it starts higher than the tree's held level). The holdout gain is small but the test shift is large, so the leaderboard can move a lot either way.

### E7. Premium smoothing

| | |
|---|---|
| **Question** | Does the strength of the block premiums' pull toward the level above matter? (E1 used 20 / 5 without choosing it) |
| **Reference** | E6 (smoothing 20 / 5): tuning 594.0, holdout 481.7 |
| **Change** | Smoothing, as rows' worth of pull: block toward 1 / block + flat type toward block |
| **Variants** | 5 / 2 · 20 / 5 · 50 / 10 · 100 / 25 |
| **Held fixed** | E6's `blend` level, `oof` premiums, settings |
| **Decision rule** | Lowest tuning RMSE |

The first cell builds the columns for E7 and E8 on each split (fit rows only); the second runs E7.

In [ ]:
# Columns for E7 (block premiums at four smoothings) and E8 (four more premiums), per split, fit rows only.
SMOOTHING = [(5, 2), (20, 5), (50, 10), (100, 25)]
GROUPS = {
    "STREET_FT_PREM": (["STREET", "FLAT_TYPE"], "REL", 20),
    "TOWN_FT_PREM": (["TOWN", "FLAT_TYPE"], "REL", 20),
    "BLOCK_FM_PREM": (["BID", "FLAT_MODEL"], "REL", 20),
    "BLOCK_FT_SQM_PREM": (["BID", "FT"], "REL_SQM", 5),
}


def group_premium(fit_df, apply_dfs, keys, target, m):
    # Smoothed mean of target per group of keys, pulled toward 1; fit rows out-of-fold.
    allk = pd.concat([fit_df[keys]] + [a[keys] for a in apply_dfs], ignore_index=True)
    codes = allk.groupby(keys, sort=False, dropna=False).ngroup().to_numpy()
    fc, ng = codes[:len(fit_df)], codes.max() + 1
    bounds = np.cumsum([len(fit_df)] + [len(a) for a in apply_dfs])
    val = fit_df[target].to_numpy()

    def enc(src, dst_codes):
        s, c = np.bincount(fc[src], val[src], ng), np.bincount(fc[src], minlength=ng)
        return (m + s[dst_codes]) / (m + c[dst_codes])

    own = np.empty(len(fit_df))
    for f_idx, p_idx in KFold(5, shuffle=True, random_state=SEED).split(fit_df):
        own[p_idx] = enc(f_idx, fc[p_idx])
    everything = np.arange(len(fit_df))
    return own, [enc(everything, codes[a:b]) for a, b in zip(bounds[:-1], bounds[1:])]


def add_more_premiums(fit_df, apply_dfs):
    for mb, mf in SMOOTHING:
        cols = premium_cols(f"oof_m{mb}_{mf}")
        own, applied = premiums(fit_df, apply_dfs, "oof", None, (mb, mf))
        fit_df[cols] = own
        for a, v in zip(apply_dfs, applied):
            a[cols] = v
    for col, (keys, target, m) in GROUPS.items():
        own, applied = group_premium(fit_df, apply_dfs, keys, target, m)
        fit_df[col] = own
        for a, v in zip(apply_dfs, applied):
            a[col] = v


for fit_df, val_df in FRAMES.values():
    add_more_premiums(fit_df, [val_df])

In [ ]:
for mb, mf in SMOOTHING:
    run2(f"smoothing {mb} / {mf}", premium_cols(f"oof_m{mb}_{mf}"), ARM)
best_m = min(SMOOTHING, key=lambda m: R2[f"smoothing {m[0]} / {m[1]}"]["tuning RMSE"])
BLOCK_COLS = premium_cols(f"oof_m{best_m[0]}_{best_m[1]}")
REF_E7 = f"smoothing {best_m[0]} / {best_m[1]}"
print(f"smoothing chosen: {best_m}")
table2([f"smoothing {m[0]} / {m[1]}" for m in SMOOTHING], f"smoothing {M_BLOCK} / {M_FT}")

**E7 findings**

- **Smoothing barely matters:** 5 / 2 vs 20 / 5 changes 0.1. Heavier smoothing is slightly worse (100 / 25: +0.3 tuning, +1.1 holdout).

**Decision: keep 5 / 2** (the rule picks it, but the gap is noise-sized). It is the reference for E8.

### E8. More premiums: other groups, per square metre

| | |
|---|---|
| **Question** | Do premiums over other groupings add anything to the block + flat type premium? |
| **Reference** | E7 (smoothing 5 / 2): tuning 593.9, holdout 481.6 |
| **Change** | + 1 premium column, out-of-fold over the fit rows, pulled toward 1 with a weight of 20 rows (5 for per sqm) |
| **Variants** | **street + flat type** and **town + flat type** (coarser groups, more rows each) · **block + flat model** (e.g. a block's DBSS or premium apartment units vs its standard ones) · **block + flat type, per sqm** (relative rent per sqm, so a block whose 4-rooms are simply larger doesn't look like a premium block) · then all that pass, together |
| **Held fixed** | E6's level, E7's block premiums, settings |
| **Decision rule** | Each is kept if it beats E7 on the tuning split; the final set is the lowest tuning RMSE among the singles and the combination |

In [ ]:
for col in GROUPS:
    run2(f"+ {col}", BLOCK_COLS + [col], ARM)
kept = [c for c in GROUPS if R2[f"+ {c}"]["tuning RMSE"] < R2[REF_E7]["tuning RMSE"]]
if len(kept) > 1:
    run2("+ " + ", ".join(kept), BLOCK_COLS + kept, ARM)
cands = [REF_E7] + [f"+ {c}" for c in GROUPS] + (["+ " + ", ".join(kept)] if len(kept) > 1 else [])
BEST_E8 = min(cands, key=lambda n: R2[n]["tuning RMSE"])
EXTRA_E8 = R2[BEST_E8]["extra"]
print(f"premiums kept: {kept or 'none'}; carried forward: {BEST_E8}")
table2(cands, REF_E7)

**E8 findings**

- **Each extra premium helps the tuning split by 0.2 to 1.0, and all four together help by nothing** (+0.1).
- Street + flat type is the best single one on the tuning split (−1.0), but it is +0.1 on the holdout. These gaps are noise-sized; the block + flat type premium already holds what these groups know.

**Decision: carry street + flat type** to E9.

### E9. Retuning LightGBM on the new columns

| | |
|---|---|
| **Question** | Do `09`'s settings, chosen without premium columns and on rent rather than rent ÷ level, still fit? |
| **Reference** | E8 with the fixed 357 rounds: holdout 481.7 |
| **Change** | LightGBM hyperparameters |
| **Variants** | The current settings + 20 random draws (`09` E2's search, slightly wider), each early-stopped (patience 100, learning rate 0.05) on the tuning split |
| **Held fixed** | E6's level, E8's columns |
| **Decision rule** | Best draw on the tuning split; refit on the holdout's fit months with its round count scaled to the larger fit set, scored once |
| **Caveat** | Rounds and draw are both picked on the tuning split, so its RMSE here is optimistic; the holdout is the fair check |

In [ ]:
N_DRAWS, MAX_ROUNDS = 20, 5000
rng = np.random.default_rng(SEED)
draw = lambda: {"num_leaves": int(rng.choice([31, 63, 127, 255, 511])),
                "min_child_samples": int(rng.choice([5, 10, 20, 50, 100, 200])),
                "subsample": float(rng.choice([0.7, 0.85, 1.0])), "subsample_freq": 1,
                "colsample_bytree": float(rng.choice([0.3, 0.5, 0.7, 0.85, 1.0])),
                "reg_lambda": float(rng.choice([0.0, 1.0, 5.0, 20.0, 50.0])),
                "cat_smooth": float(rng.choice([1.0, 10.0, 30.0]))}
current = {k: v for k, v in LGBM_TUNED.items() if k not in ("n_estimators", "learning_rate")}
tune_fit_df, tune_val_df = FRAMES["tuning"]
y_tv = tune_val_df[TARGET].to_numpy()
rows, t0 = [], time.time()
for i, params in enumerate([current] + [draw() for _ in range(N_DRAWS)]):
    (p,), rounds = fit_arm(tune_fit_df, [tune_val_df], EXTRA_E8, ARM,
                           {**params, "n_estimators": MAX_ROUNDS, "learning_rate": 0.05}, early_stop=True)
    rows.append({"draw": i, "tuning RMSE": rmse(y_tv, p), "rounds": rounds, "params": params})
search_e9 = pd.DataFrame(rows).sort_values("tuning RMSE")
print(f"search time: {(time.time() - t0) / 60:.1f} min")
search_e9.head(5)

In [ ]:
best_row = search_e9.iloc[0]
scale_h = len(FRAMES["holdout"][0]) / len(tune_fit_df)
TUNED_E9 = {**best_row["params"], "learning_rate": 0.05}
cur_row = search_e9[search_e9["draw"] == 0].iloc[0]
fit_df, val_df = FRAMES["holdout"]
y = val_df[TARGET].to_numpy()
retune_tbl = []
for label, params, rounds in [("current settings, early-stopped rounds", {**current, "learning_rate": 0.05}, cur_row["rounds"]),
                              ("retuned", TUNED_E9, best_row["rounds"])]:
    (p,) = fit_arm(fit_df, [val_df], EXTRA_E8, ARM, {**params, "n_estimators": int(round(rounds * scale_h))})
    retune_tbl.append({"settings": label, "tuning RMSE (early-stopped)": cur_row["tuning RMSE"] if label.startswith("current") else best_row["tuning RMSE"],
                       "holdout RMSE": rmse(y, p), "holdout bias": float(np.mean(p - y)), "rounds (holdout)": int(round(rounds * scale_h))})
display(pd.DataFrame(retune_tbl).set_index("settings"))
print(f"E8 row with the fixed 357 rounds: holdout {R2[BEST_E8]['holdout RMSE']:.2f}")
print("retuned settings:", TUNED_E9, "| rounds on the tuning split:", int(best_row["rounds"]))

**E9 findings**

- **Retuning gains under 1:** 481.7 (fixed 357 rounds) → 481.1 with the current settings and early-stopped rounds → 480.8 retuned. The best draw (31 leaves, 5 rows per leaf, row subsample 0.7) beats the current settings on the tuning split by only 0.06.
- As in `09`, the model's settings are not where the error is.

**Decision: keep the retuned settings** for S3.

### S3. Final model and submission

| | |
|---|---|
| **Model** | E6's `blend` level × LightGBM premium, E7's block premiums (5 / 2) + E8's street + flat type premium, E9's settings |
| **Refit** | All of train, rounds = E9's tuning-split round count × all train ÷ tuning fit |
| **File** | `outputs/submission_10_final.csv` |

The table after it traces the holdout RMSE along the path, with each checkpoint's file and public leaderboard
score where one is recorded.

In [ ]:
add_more_premiums(full_tr, [full_te])
rounds_all = int(round(best_row["rounds"] * len(full_tr) / len(tune_fit_df)))
(test_final,) = fit_arm(full_tr, [full_te], EXTRA_E8, ARM, {**TUNED_E9, "n_estimators": rounds_all})
assert len(test_final) == len(te) and np.isfinite(test_final).all()
SUB_FINAL = OUT / "submission_10_final.csv"
pd.DataFrame({"Id": np.arange(len(test_final)), "Predicted": test_final}).to_csv(SUB_FINAL, index=False)
print(f"wrote {SUB_FINAL}: {ARM}, {len(EXTRA_E8)} premium columns, {rounds_all} rounds")

steps = pd.DataFrame([
    {"step": "E0: 09 tuned LightGBM", "holdout RMSE": RESULTS["base (09 tuned LightGBM)"]["holdout RMSE"], "file": "submission_09_lgbm.csv", "leaderboard": 443.25},
    {"step": "S1: E5 (+ block premiums)", "holdout RMSE": R2[REF]["holdout RMSE"], "file": "submission_10_features.csv", "leaderboard": 436.8},
    {"step": f"S2: E6 ({ARM})", "holdout RMSE": R2[f"E5 final, {ARM}"]["holdout RMSE"], "file": "submission_10_level.csv", "leaderboard": np.nan},
    {"step": "E8: more premiums", "holdout RMSE": R2[BEST_E8]["holdout RMSE"], "file": "-", "leaderboard": np.nan},
    {"step": "S3: E9 retuned", "holdout RMSE": retune_tbl[1]["holdout RMSE"], "file": "submission_10_final.csv", "leaderboard": 439.48},
]).set_index("step")
steps

**S3 notes**

- **Holdout path: 487.5 (E0) → 483.1 (E1) → 481.7 (E6) → 480.8 (E8 + E9).** Locally, the block premiums and the level are the only steps worth more than 1.
- **Leaderboard disagrees with the holdout on the level:** S3 scored 439.5, worse than S1's 436.8, although its
  holdout is 2.3 better. S2 (the level alone) has no recorded score, but S3 differs from S1 mainly in level (about
  +87 on average; Appendix B), so the `blend` level overshoots on the test months. The validated submission is S1.

## Summary

| ID | Experiment | Tuning Δ | Holdout Δ | Decision |
|---|---|---|---|---|
| E1 | Block + flat type premium (`oof`) | −3.8 | −4.4 | **kept** |
| E2 | Coordinates | −0.5 | +0.2 | dropped in E5 |
| E3 | Neighbouring blocks | −1.2 | −1.0 | dropped in E5 |
| E4 | Recency weighting | +1.2 or worse | +1.2 or worse | dropped |
| E5 | E1 + E2 + E3 | −3.0 | −4.2 | E1 alone kept |
| E6 | Market level × premium (`blend`) | −5.8 | −1.5 | **kept** |
| E7 | Premium smoothing 5 / 2 | −0.1 | −0.1 | kept (noise) |
| E8 | + street + flat type premium | −1.0 | +0.1 | kept (noise) |
| E9 | Retuned LightGBM | n/a | −0.9 | kept |

- **Block + flat type premium (E1): the one feature that clearly helps.** −3.8 tuning, −4.4 holdout, −6.4 on the leaderboard (S1: 443.2 → 436.8).
- **Market level × premium with the blend forecast (E6): −5.8 tuning, −1.5 holdout,** and it removes most of the under-prediction 10 to 15 months ahead. It shifts test predictions up by about 90 on average, and on the leaderboard the final file that includes it (S3, 439.5) is worse than S1: rents rose over the test window, but by less than the blend forecast (Appendix B).
- **Neighbouring blocks (E3) about −1; coordinates, recency weighting, smoothing, extra premiums and retuning (E2, E4, E7, E8, E9) all within ±1, or worse.**
- **PCA and graph networks were not built:** the location signal they would add is already in `TOWN`, `DIST` and the premiums, and the neighbour premium (the cheap graph idea) is worth about 1.
- **What is left:** about 525 RMSE even 1 to 3 months ahead. That is rent differences between similar flats that the columns cannot explain, and no amount of tuning is touching it.
- **No leakage:** nothing above uses test rents or leaderboard scores to make a choice; leaderboard numbers are only reported. The leaderboard-tuned mix (434.5) is Appendix B.
- **Files:** `outputs/submission_10_features.csv` (S1, 436.8), `outputs/submission_10_level.csv` (S2), `outputs/submission_10_final.csv` (S3, 439.5).

## Appendix A. Test rents from data.gov.sg's published rental records

**This is not a valid model: it reads the test labels.**

| | |
|---|---|
| **Question** | Can the test rents be looked up instead of forecast? |
| **Uses** | data.gov.sg's "Renting out of flats" (from 2021-01, updated monthly): approval month, town, block, street, flat type, monthly rent. The competition is drawn from it, and the test window (2025-03 to 2026-07) is already published |
| **Why it is not valid** | The outside file contains the test labels. CS5228 allows outside data, but this is a lookup, not a model |
| **Checked on** | The last 15 training months, treated as unlabelled |
| **Result** | Group mean 188.4 vs 504.9 for a model on the same rows; it explains the leaderboard's top scores (173 to 179) |

Nothing in the main part uses it. The frames here are lower case and separate from the main part's.

A.1 Download the published records · A.2 Clean and match streets · A.3 Check on the last 15 training months ·
A.4 Submission · A.5 Summary

### A.1 Download the published records

data.gov.sg dataset `d_c9f57187485a850908655db0e8cfe651`. The open-data API hands out a short-lived download
link; the request is retried because the API rate-limits. The file is saved to `data/external/` and reused on
later runs, so the numbers below stay fixed unless the file is deleted (data.gov.sg adds a month at a time).

In [ ]:
DATASET_ID = "d_c9f57187485a850908655db0e8cfe651"
EXT_FILE = EXTERNAL / "renting_out_of_flats.csv"


def download(dataset_id, dest, tries=6, wait=10):
    api = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/initiate-download"
    for _ in range(tries):
        with urllib.request.urlopen(api) as r:
            url = (json.load(r).get("data") or {}).get("url")
        if url:
            urllib.request.urlretrieve(url, dest)
            return
        time.sleep(wait)  # rate-limited: no link this time
    raise RuntimeError("data.gov.sg did not return a download link")


EXTERNAL.mkdir(parents=True, exist_ok=True)
if not EXT_FILE.exists():
    download(DATASET_ID, EXT_FILE)
ext_raw = pd.read_csv(EXT_FILE)
print(f"{EXT_FILE}: {len(ext_raw):,} rows, {ext_raw['rent_approval_date'].min()} to {ext_raw['rent_approval_date'].max()}")
ext_raw.head(3)

### A.2 Clean and match streets

Both sides are lowercased and flat types normalised ("4 room" = "4-room"), as in `07`/`09`. The competition
files spell street names out ("ang mo kio avenue 4"); data.gov.sg abbreviates them ("ang mo kio ave 4"). Each
of our town + block + street combinations is matched to the data.gov.sg street on the same town and block whose
expanded name is identical; a similarity score is the fallback, and the check below reports if it was ever
needed.

In [ ]:
def norm(df, street):
    out = df.copy()
    for c in ["town", "block", street, "flat_type"]:
        out[c] = out[c].astype(str).str.strip().str.lower()
    out["flat_type"] = out["flat_type"].str.replace(r"[\s\-]+", "-", regex=True)
    return out


ext = norm(ext_raw, "street_name")
train = norm(pd.read_csv(DATA / "train.csv").rename(columns=str.lower), "street")
test = norm(pd.read_csv(DATA / "test.csv").rename(columns=str.lower), "street")

per_month = pd.DataFrame({
    "data.gov.sg": ext["rent_approval_date"].value_counts(),
    "competition": pd.concat([train, test])["rent_approval_date"].value_counts(),
}).dropna()
print(f"competition rows as a share of data.gov.sg's, per month: "
      f"{(per_month['competition'] / per_month['data.gov.sg']).min():.1%} to {(per_month['competition'] / per_month['data.gov.sg']).max():.1%}")

In [ ]:
import difflib

ABBREV = {
    "ave": "avenue", "st": "street", "rd": "road", "dr": "drive", "cres": "crescent", "nth": "north", "sth": "south",
    "ctrl": "central", "c'wealth": "commonwealth", "bt": "bukit", "jln": "jalan", "lor": "lorong", "tg": "tanjong",
    "upp": "upper", "pl": "place", "cl": "close", "ctr": "centre", "pk": "park", "ter": "terrace", "hts": "heights",
    "gdns": "gardens", "kg": "kampong", "mkt": "market", "ct": "court", "sq": "square", "blvd": "boulevard",
    "hwy": "highway", "cct": "circuit", "vw": "view", "lk": "link", "gr": "grove", "ln": "lane", "e": "east", "w": "west",
}


def expand(street):
    return " ".join(ABBREV.get(t, t) for t in street.replace("st.", "saint").split())


ours = pd.concat([train, test])[["town", "block", "street"]].drop_duplicates()
candidates = ext[["town", "block", "street_name"]].drop_duplicates()
candidates["expanded"] = candidates["street_name"].map(expand)
pairs = ours.merge(candidates, on=["town", "block"], how="left")
pairs["exact"] = pairs["expanded"] == pairs["street"]
pairs["similarity"] = [difflib.SequenceMatcher(None, a, str(b)).ratio() for a, b in zip(pairs["street"], pairs["expanded"])]
street_map = (pairs.sort_values(["exact", "similarity"], ascending=False)
              .drop_duplicates(["town", "block", "street"])[["town", "block", "street", "street_name", "exact", "similarity"]])
street_map.to_csv(EXTERNAL / "street_map.csv", index=False)

print(f"our town + block + street combinations: {len(street_map):,}")
print(f"  matched exactly after expanding abbreviations: {street_map['exact'].sum():,}")
print(f"  matched by similarity only: {(~street_map['exact'] & street_map['street_name'].notna()).sum():,}")
print(f"  no data.gov.sg street on that town + block: {street_map['street_name'].isna().sum():,}")

train = train.merge(street_map[["town", "block", "street", "street_name"]], on=["town", "block", "street"], how="left", validate="m:1")
test = test.merge(street_map[["town", "block", "street", "street_name"]], on=["town", "block", "street"], how="left", validate="m:1")
assert train["street_name"].notna().all() and test["street_name"].notna().all()

### A.3 Check on the last 15 training months

The 15 months `09` uses as its holdout (2024-01 to 2025-03) are treated as unlabelled, and their rents are looked
up from data.gov.sg the same way the test rents will be. A **group** is one block + flat type + month: data.gov.sg
has no floor area or flat model, so rows in the same group cannot be told apart from its side.

Ways to fill each row:

- **group mean** or **group median** of the data.gov.sg rents;
- **rank-assigned:** order the group's rows by a model's prediction and hand out the data.gov.sg rents in the same
  order (when data.gov.sg has more rows than the group, the order-preserving choice closest to the predictions);
- **mixes** of the group mean with the rank-assigned rents or with the model's within-group differences.

The model is LightGBM on the competition's columns with block and street, scored out-of-fold (5 random folds
over all of train), so each row's prediction comes from a model that did not see its rent.

In [ ]:
G = ["town", "block", "street_name", "flat_type", "rent_approval_date"]
CHECK_START = "2024-01"
CATS = ["town", "flat_type", "flat_model", "block", "street_name"]
LEVELS = {c: sorted(pd.concat([train[c], test[c]]).unique()) for c in CATS}
MODEL_COLS = CATS + ["floor_area_sqm", "lease_commence_date", "month"]


def model_matrix(df):
    X = df.assign(month=pd.to_datetime(df["rent_approval_date"]).dt.year * 12 + pd.to_datetime(df["rent_approval_date"]).dt.month)[MODEL_COLS]
    for c in CATS:
        X[c] = pd.Categorical(X[c], categories=LEVELS[c])
    return X


X, y_all = model_matrix(train), train["monthly_rent"].to_numpy()
oof = np.empty(len(train))
for fit_idx, pred_idx in KFold(5, shuffle=True, random_state=SEED).split(X):
    est = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=127, verbose=-1, random_state=SEED)
    oof[pred_idx] = est.fit(X.iloc[fit_idx], y_all[fit_idx]).predict(X.iloc[pred_idx])

In [ ]:
def assign(pred, pool):
    # Pick len(pred) rents from pool in the same order as pred, minimising the squared gap (order-preserving DP).
    pred = np.asarray(pred, float)
    order = np.argsort(pred)
    ps, pool = pred[order], np.sort(np.asarray(pool, float))
    n, k = len(ps), len(pool)
    if k == n:
        chosen = pool
    elif k < n:
        chosen = np.quantile(pool, (np.arange(n) + 0.5) / n)
    else:
        D = np.full((n + 1, k + 1), np.inf)
        D[0, :] = 0
        for i in range(1, n + 1):
            for j in range(i, k + 1):
                D[i, j] = min(D[i, j - 1], D[i - 1, j - 1] + (ps[i - 1] - pool[j - 1]) ** 2)
        chosen, i, j = np.empty(n), n, k
        while i > 0:
            if j > i and D[i, j] == D[i, j - 1]:
                j -= 1
            else:
                chosen[i - 1] = pool[j - 1]
                i, j = i - 1, j - 1
    out = np.empty(n)
    out[order] = chosen
    return out


def lookup(rows, pool_df, pred):
    pools = pool_df.groupby(G)["monthly_rent"].apply(np.asarray)
    mean, median, ranked = (np.full(len(rows), np.nan) for _ in range(3))
    for key, idx in rows.groupby(G).indices.items():
        pool = pools.get(key)
        if pool is not None:
            mean[idx], median[idx], ranked[idx] = pool.mean(), np.median(pool), assign(pred[idx], pool)
    return mean, median, ranked


in_check = train["rent_approval_date"].ge(CHECK_START).to_numpy()
check = train[in_check].reset_index(drop=True)
y, pred = check["monthly_rent"].to_numpy(), oof[in_check]
mean, median, ranked = lookup(check, ext[ext["rent_approval_date"].ge(CHECK_START)], pred)
print(f"check rows with a data.gov.sg group: {np.isfinite(mean).mean():.2%}")

group_pred = pd.Series(pred).groupby([check[c] for c in G]).transform("mean").to_numpy()
rmse = lambda p: float(np.sqrt(np.mean((p - y) ** 2)))
results = {"model alone (out-of-fold LightGBM)": rmse(pred), "group mean": rmse(mean), "group median": rmse(median),
           "rank-assigned": rmse(ranked)}
results.update({f"group mean x {1 - a:.1f} + rank-assigned x {a:.1f}": rmse((1 - a) * mean + a * ranked) for a in (0.2, 0.4)})
results.update({f"group mean + {b} x model's within-group difference": rmse(mean + b * (pred - group_pred)) for b in (0.25, 0.5, 1.0)})
pd.Series(results, name="RMSE, 2024-01 to 2025-03").to_frame()

In [ ]:
size = check.groupby(G)["town"].transform("size").clip(upper=5).rename("rows in group (5 = 5+)")
pd.DataFrame({"share of rows": size.value_counts(normalize=True).sort_index(),
              "group-mean RMSE": pd.Series((mean - y) ** 2).groupby(size).mean() ** 0.5,
              "exact rent, rank-assigned": pd.Series(ranked == y).groupby(size).mean()})

- **Every check row has a data.gov.sg group, and the group mean scores 188.4**, against 504.9 for the model alone
  on the same rows (`09`'s time-ordered holdout: about 490).
- **72% of rows are alone in their group** (the only rental of that flat type in that block that month). The lookup
  gets 99% of those exactly; their RMSE is 35.6, not 0, because the competition dropped 3 to 5% of data.gov.sg's
  rows, so a group of one on our side can hold two rents on theirs.
- **All the remaining error is in groups of 2+** (RMSE 337 to 428). data.gov.sg has no floor level, floor area or
  flat model, and our columns do not separate flats in the same block either: rank-assigning gets only 20 to 32%
  of those rows exactly, and a wrong pick swaps whole rents, so it scores 335 overall.
- **Nothing beats the plain group mean.** Every mix with the model's ordering or its within-group differences is
  worse (195.6 to 240.4). The mean is what minimises squared error when the rows in a group cannot be told apart.

### A.4 Submission

Each test row gets the mean data.gov.sg rent of its group, over the test months. 2025-03 is in both train and
test, so for that month the rents already used by training rows are taken out of each group's pool first. A row
with no group would fall back to `09`'s blend.

Format as Kaggle requires: `Id` (0-based row number in `test.csv`) and `Predicted`.

In [ ]:
pool = ext[ext["rent_approval_date"].ge(test["rent_approval_date"].min())].copy()
# Drop one pool rent for every training row with the same group and rent (only 2025-03 overlaps).
pool["nth"] = pool.groupby(G + ["monthly_rent"]).cumcount()
used = train.groupby(G + ["monthly_rent"]).size().rename("used").reset_index()
pool = pool.merge(used, on=G + ["monthly_rent"], how="left").fillna({"used": 0})
pool = pool[pool["nth"] >= pool["used"]]

group_mean = pool.groupby(G)["monthly_rent"].mean().rename("group_mean").reset_index()
filled = test.reset_index().merge(group_mean, on=G, how="left").sort_values("index")["group_mean"].to_numpy()
fallback = pd.read_csv(OUT / "submission_09_blend.csv")["Predicted"].to_numpy()
missing = np.isnan(filled)
test_pred = np.where(missing, fallback, filled)
assert len(test_pred) == len(test) and np.isfinite(test_pred).all()

SUBMISSION = OUT / "submission_10_external.csv"
pd.DataFrame({"Id": np.arange(len(test_pred)), "Predicted": test_pred}).to_csv(SUBMISSION, index=False)
print(f"wrote {SUBMISSION}: {len(test_pred):,} rows, {missing.sum()} from the fallback model")
print(f"RMSE between this and 09's blend (how far the lookup moves the predictions): "
      f"{np.sqrt(np.mean((test_pred - fallback) ** 2)):.1f}")
months = test["rent_approval_date"].to_numpy()
pd.DataFrame({"median, lookup": pd.Series(test_pred, index=months).groupby(level=0).median(),
              "median, 09 blend": pd.Series(fallback, index=months).groupby(level=0).median()}).iloc[::3]

- `outputs/submission_10_external.csv`: 50,000 rows, all filled from data.gov.sg; the fallback was not needed.
- **The lookup puts test rents about $50 higher than `09`'s blend by 2026** (median 3,200 vs 3,135 to 3,150). Rents
  kept rising slowly through the test window, which is the under-prediction `09` §7 measured.

### A.5 Summary

- **Outside data is allowed in CS5228**, and data.gov.sg's "Renting out of flats" is the source of the competition's
  rows: the competition holds 95 to 97% of its rows in every month, and all 9,456 town + block + street combinations
  match once its street abbreviations are expanded.
- **Looking up each test row's block + flat type + month mean rent scores 188.4** on the last 15 training months,
  against about 490 for the best model in `09`. The leaderboard's top three are at 173 to 179.
- **The error left is within-group:** rows sharing a block, flat type and month that nothing here can tell apart.
  Model-based orderings and adjustments make it worse, so the submission uses the plain group mean.
- **Submission:** `outputs/submission_10_external.csv` (`Id`, `Predicted`).
- **Reproducibility:** the download is saved to `data/external/renting_out_of_flats.csv` and reused; delete it to
  fetch the latest release from data.gov.sg.

## Appendix B. Mixing two submissions using leaderboard scores

**This uses the public leaderboard to choose a number, so its score is optimistic.**

| | |
|---|---|
| **Question** | What mix of S1 and S3 would the public leaderboard prefer? |
| **Uses** | The two files' public leaderboard scores (S1 436.80, S3 439.48). No test rent is seen |
| **Why it is not valid** | The weight w is fitted to the public leaderboard's rows, so the private leaderboard will probably show a smaller gain |

S1 (`submission_10_features.csv`) and S3 (`submission_10_final.csv`) differ mostly in level: the final file is about
87 higher on average. Mixing them as (1 − w) × S1 + w × S3 moves along the line between the two files, and along
that line the squared error is a parabola in w:

  MSE(w) = MSE(0) + 2w·E[d·e] + w²·E[d²], with d = S3 − S1 and e = S1 − actual.

MSE(0) and MSE(1) are the two leaderboard scores squared, and E[d²] comes from the files, so E[d·e] and the best w
follow without uploading anything.

In [ ]:
LB = {"features": 436.80491, "final": 439.48443}  # public leaderboard scores of the two files
f_pred = pd.read_csv(OUT / "submission_10_features.csv")["Predicted"].to_numpy()
l_pred = pd.read_csv(OUT / "submission_10_final.csv")["Predicted"].to_numpy()
d = l_pred - f_pred
Ed2 = np.mean(d ** 2)
Ede = (LB["final"] ** 2 - LB["features"] ** 2 - Ed2) / 2
w = -Ede / Ed2
predicted = np.sqrt(LB["features"] ** 2 + 2 * w * Ede + w ** 2 * Ed2)
print(f"final - features: mean {d.mean():+.1f}, RMS {np.sqrt(Ed2):.1f}")
print(f"best weight on final: {w:.3f}  -> predicted public leaderboard RMSE {predicted:.2f}")
W_MIX = 0.4
mix = (1 - W_MIX) * f_pred + W_MIX * l_pred
SUB_MIX = OUT / "submission_10_mix.csv"
pd.DataFrame({"Id": np.arange(len(mix)), "Predicted": mix}).to_csv(SUB_MIX, index=False)
print(f"wrote {SUB_MIX} with weight {W_MIX} on final")

- **The best weight is about 0.4, predicted at about 434.5, and the uploaded mix scored that** (from 436.8).
- **Reading:** rents did rise over the test window, so the `MONTH_INDEX` model (S1) is too low, but only by about
  40% of what the blend level forecast (+2.1%), so the level model overshoots. On the holdout the blend also
  over-predicted in the first months (+24).
- **Not a valid result for the report:** w was chosen with test-side feedback. The validated path is S1's 436.8.

In [ ]:
print(f"runtime: {(time.time() - T0) / 60:.1f} min")

## Part 2.4. Graph neural networks

Blocks are nodes, edges join nearby blocks (`src/hdb_rentals/amenity_graph.py`, `gnn.py`). Each step runs a script from `scripts/` and shows its summary. Order matters: the last step reads the previous one's predictions.

In [ ]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def run_script(name, *args):
    """Run a GNN script in its own process (keeps torch and LightGBM out of this kernel)."""
    subprocess.run([sys.executable, str(ROOT / "scripts" / name), *args], cwd=ROOT, check=True)

### Edge weights: uniform, distance, similarity, learned, multiscale

`scripts/experiment_weighted_gnns.py`: the same network with each edge weighting, against `MLP_matched` (the graph switched off). Writes `outputs/12_weighted_gnns/`. Optional: set `RUN_EDGE_WEIGHT_EXPERIMENT = True` to run it.

In [ ]:
# About an hour on a laptop CPU; its results are not used elsewhere, so it is off by default.
RUN_EDGE_WEIGHT_EXPERIMENT = False
if RUN_EDGE_WEIGHT_EXPERIMENT:
    run_script("experiment_weighted_gnns.py")
    display(pd.read_csv(ROOT / "outputs/12_weighted_gnns/seed_metrics.csv"))

### GNN vs Part 2.3's LightGBM, on the same splits

`scripts/compare_gnn_nb10.py`: re-runs Part 2.3's E0 LightGBM (must reproduce 603.6 / 487.5), then `GNN_uniform` and `MLP_matched`. Writes `outputs/gnn_vs_nb10/`.

In [ ]:
run_script("compare_gnn_nb10.py")
pd.read_csv(ROOT / "outputs/gnn_vs_nb10/summary.csv")

### GNN with block premiums, and GNN + LightGBM blends

`scripts/gnn_premiums_blend.py`. Writes `outputs/gnn_premiums_blend/`.

In [ ]:
run_script("gnn_premiums_blend.py")
pd.read_csv(ROOT / "outputs/gnn_premiums_blend/summary.csv")

### Neighbours' premiums through the graph

`scripts/gnn_neighbour_premiums.py` (reads the previous step's predictions). Writes `outputs/gnn_neighbour_premiums/`.

In [ ]:
run_script("gnn_neighbour_premiums.py")
pd.read_csv(ROOT / "outputs/gnn_neighbour_premiums/summary.csv")